# AGRI-INTELLIGENCE | PLATFORM NOTEBOOK V1
**ArqTech Labs · Multimodal Agricultural Intelligence**

Este notebook é a interface de orquestração no Google Colab. A lógica reutilizável permanece em `src/agri_intelligence`; dados, estado e artefatos persistem no Google Drive.

**Milestone atual:** bootstrap persistente e recuperação verificável das fontes da Phase 2.5. Nenhum treinamento ou inferência é executado nesta versão inicial.

Branch de desenvolvimento: `feat/platform-notebook-v1`.

## 0. Regras de execução

- Não instalar dependências automaticamente.
- Não baixar pesos nem iniciar treinamento/inferência nesta etapa.
- Não carregar ortomosaicos completos na RAM; usar leitura por janelas.
- Recuperar arquivos somente quando ausentes ou com checksum divergente.
- Preservar arquivos inválidos como backup antes de substituí-los.
- Interromper em checksum divergente, fonte inesperada ou branch incorreta.
- Cada etapa registra configuração, commit, artefatos e resultado.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import os, sys, json, hashlib, subprocess, platform

PROJECT_NAME = "Agri-Intelligence"
REPOSITORY = "edu-moraess/agri-intelligence"
REPOSITORY_URL = f"https://github.com/{REPOSITORY}.git"
DEVELOPMENT_BRANCH = "feat/platform-notebook-v1"
ZENODO_RECORD_ID = "15235602"
ZENODO_API = f"https://zenodo.org/api/records/{ZENODO_RECORD_ID}"

DRIVE_ROOT = Path("/content/drive/MyDrive/ArqTechLabs/Agri-Intelligence")
REPO_ROOT = Path("/content/agri-intelligence-platform-v1")
SOURCE_ROOT = DRIVE_ROOT / "data/raw/zenodo_15235602"
INTERIM_ROOT = DRIVE_ROOT / "data/interim/phase_2_5"
PROCESSED_ROOT = DRIVE_ROOT / "data/processed/phase_2_5"
STATE_ROOT = DRIVE_ROOT / "state"
ARTIFACT_ROOT = DRIVE_ROOT / "artifacts"
LOG_ROOT = DRIVE_ROOT / "logs"

EXPECTED_RASTER_MD5 = {
    "orthomosaic_1.tif": "062a5c9028fff72f3fc276917e0a5af1",
    "orthomosaic_2.tif": "c5c478aa73a22f11d52eb8176c4c8b1a",
    "orthomosaic_3.tif": "a800e5be8ec95f4a61d2c24a0b4e4a74",
}
EXPECTED_RASTER_SHA256 = {
    "orthomosaic_1.tif": "d4317d8eb402a15d33e084e131de26bd6340a14756148b4e6011a912f3695a2c",
    "orthomosaic_2.tif": "c5a81cfab297866a82633130f161ac3a31524956a17c9e94e98f8e6ac3583feb",
    "orthomosaic_3.tif": "86abb6bb4dca079aa07de0520e376e34fcaf21efaba93cfd587c520a4008948d",
}
EXPECTED_GT_COUNTS = {"gt_1": 61251, "gt_2": 30547, "gt_3": 55374}

print(f"{PROJECT_NAME} | PLATFORM V1")
print("Branch alvo:", DEVELOPMENT_BRANCH)
print("Drive:", DRIVE_ROOT)
print("Fonte:", ZENODO_API)

## 1. Montagem do armazenamento persistente

Os arquivos brutos e resultados ficam no Drive, não no disco efêmero de `/content`.

In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

for path in [DRIVE_ROOT, SOURCE_ROOT, INTERIM_ROOT, PROCESSED_ROOT,
             STATE_ROOT, ARTIFACT_ROOT, ARTIFACT_ROOT / "manifests", LOG_ROOT]:
    path.mkdir(parents=True, exist_ok=True)

print("PERSISTENT STORAGE: READY")
print("Source root:", SOURCE_ROOT)
print("State root:", STATE_ROOT)

## 2. Repositório e identidade da execução

Em uma sessão nova, o notebook clona a branch de desenvolvimento. Se o repositório já existir, não troca de branch nem sobrescreve alterações locais.

In [ ]:
def run_cmd(args, cwd=None):
    result = subprocess.run(args, cwd=cwd, text=True, capture_output=True)
    if result.returncode != 0:
        raise RuntimeError(result.stderr.strip() or result.stdout.strip())
    return result.stdout.strip()

if not (REPO_ROOT / ".git").exists():
    if REPO_ROOT.exists() and any(REPO_ROOT.iterdir()):
        raise RuntimeError(f"Diretório existe e não é um clone Git limpo: {REPO_ROOT}")
    run_cmd(["git", "clone", "--branch", DEVELOPMENT_BRANCH,
             "--single-branch", REPOSITORY_URL, str(REPO_ROOT)])

current_branch = run_cmd(["git", "branch", "--show-current"], cwd=REPO_ROOT)
current_commit = run_cmd(["git", "rev-parse", "HEAD"], cwd=REPO_ROOT)
working_tree = run_cmd(["git", "status", "--porcelain"], cwd=REPO_ROOT)

if current_branch != DEVELOPMENT_BRANCH:
    raise RuntimeError(
        f"Branch incorreta: {current_branch}. Esperada: {DEVELOPMENT_BRANCH}. "
        "Não foi feito checkout automático."
    )

sys.path.insert(0, str(REPO_ROOT / "src"))
print("BRANCH:", current_branch)
print("HEAD:", current_commit)
print("WORKTREE:", "CLEAN" if not working_tree else "MODIFIED")
if working_tree:
    print(working_tree)

### 2.1. Recuperar o checkpoint local da Phase 2.5

O bundle criado anteriormente no Drive pode conter o protocolo e os módulos L2 que não foram publicados no GitHub. Esta célula verifica e importa a referência para o clone novo **sem trocar de branch, fazer merge ou alterar a árvore de trabalho**.

In [ ]:
BUNDLE_PATH = Path("/content/drive/MyDrive/agri_phase25/phase25_protocol.bundle")
BUNDLE_REF = "refs/heads/feat/phase-2-5-first-real-experiment"

if not BUNDLE_PATH.is_file():
    print("PHASE 2.5 BUNDLE: NOT FOUND")
    print("A continuidade poderá ser reconstruída a partir da base remota, sem presumir que L2 esteja presente.")
else:
    verification = subprocess.run(
        ["git", "bundle", "verify", str(BUNDLE_PATH)],
        cwd=REPO_ROOT, text=True, capture_output=True
    )
    if verification.returncode != 0:
        raise RuntimeError("Bundle Phase 2.5 inválido: " + verification.stderr.strip())

    refs_output = run_cmd(["git", "bundle", "list-heads", str(BUNDLE_PATH)], cwd=REPO_ROOT)
    bundle_refs = [line.split()[-1] for line in refs_output.splitlines() if line.strip()]
    if BUNDLE_REF not in bundle_refs:
        raise RuntimeError(f"Referência esperada não encontrada no bundle. Refs: {bundle_refs}")

    run_cmd([
        "git", "fetch", str(BUNDLE_PATH),
        f"{BUNDLE_REF}:refs/remotes/backup/phase25"
    ], cwd=REPO_ROOT)
    bundle_commit = run_cmd(
        ["git", "rev-parse", "refs/remotes/backup/phase25"], cwd=REPO_ROOT
    )
    print("PHASE 2.5 BUNDLE: VERIFIED")
    print("Imported ref: refs/remotes/backup/phase25")
    print("Bundle commit:", bundle_commit)
    print("Current branch unchanged:", current_branch)
    bundle_state = {
        "bundle_path": str(BUNDLE_PATH),
        "source_ref": BUNDLE_REF,
        "imported_ref": "refs/remotes/backup/phase25",
        "commit": bundle_commit,
        "verified_at_utc": datetime.now(timezone.utc).isoformat(),
    }
    bundle_state_path = STATE_ROOT / "phase_2_5_bundle_recovery.json"
    bundle_state_temp = bundle_state_path.with_name(bundle_state_path.name + ".part")
    bundle_state_temp.write_text(json.dumps(bundle_state, indent=2), encoding="utf-8")
    bundle_state_temp.replace(bundle_state_path)

## 3. Compatibilidade do ambiente

Esta etapa apenas detecta dependências. Não instala nem atualiza pacotes.

In [ ]:
import importlib.util
import sys

OPTIONAL_MODULES = ["numpy", "pandas", "rasterio", "geopandas", "shapely", "requests", "PIL"]
availability = {name: importlib.util.find_spec(name) is not None for name in OPTIONAL_MODULES}

runtime = {
    "python": sys.version,
    "platform": platform.platform(),
    "modules": availability,
    "branch": current_branch,
    "commit": current_commit,
    "checked_at_utc": datetime.now(timezone.utc).isoformat(),
}

STATE_ROOT.mkdir(parents=True, exist_ok=True)
(STATE_ROOT / "runtime_environment.json").write_text(
    json.dumps(runtime, indent=2), encoding="utf-8"
)

for name, available in availability.items():
    print(f"{'AVAILABLE' if available else 'MISSING'} | {name}")
print("Python:", sys.version.split()[0])
print("Nenhuma instalação automática foi executada.")

## 4. Recuperação persistente das fontes — Zenodo 15235602

O conjunto contém três GeoTIFFs e três famílias Shapefile. Cada Shapefile depende de arquivos auxiliares (`.shp`, `.shx`, `.dbf`, `.prj`, `.cpg`; e metadados `.qmd` quando publicados). O processo abaixo seleciona somente os ortomosaicos e GT. Não utiliza `HC_bboxes_*` nem `tiles_*` como ground truth.

A recuperação é retomável: arquivos válidos são preservados; downloads são gravados como `.part`, verificados por tamanho e MD5 do registro e só então renomeados. SHA-256 é calculado para a proveniência local.

In [ ]:
import requests, shutil, time

def digest_file(path, algorithm="sha256", chunk_size=8 * 1024 * 1024):
    h = hashlib.new(algorithm)
    with open(path, "rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()

def atomic_json(path, payload):
    path = Path(path)
    temp = path.with_name(path.name + ".part")
    temp.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding="utf-8")
    temp.replace(path)

response = requests.get(ZENODO_API, timeout=(30, 120))
response.raise_for_status()
record = response.json()
if str(record.get("id")) != ZENODO_RECORD_ID:
    raise RuntimeError("O registro Zenodo retornado não corresponde ao ID solicitado.")

raw_files = record.get("files", [])
if isinstance(raw_files, dict):
    if isinstance(raw_files.get("entries"), list):
        raw_files = raw_files["entries"]
    else:
        raw_files = list(raw_files.values())
file_index = {item.get("key") or item.get("filename"): item for item in raw_files}

core_names = [f"orthomosaic_{i}.tif" for i in (1, 2, 3)]
gt_extensions = [".shp", ".shx", ".dbf", ".prj", ".cpg"]
gt_names = [f"gt_{i}{ext}" for i in (1, 2, 3) for ext in gt_extensions]
optional_names = [f"gt_{i}.qmd" for i in (1, 2, 3)]
required_names = core_names + gt_names
missing_metadata = sorted(set(required_names) - set(file_index))
if missing_metadata:
    raise RuntimeError(f"Arquivos obrigatórios ausentes no registro: {missing_metadata}")

metadata = record.get("metadata", {})
license_metadata = metadata.get("license") or metadata.get("rights")
print("Registro:", record.get("doi") or f"Zenodo {ZENODO_RECORD_ID}")
print("Título:", metadata.get("title", "não informado"))
print("Licença declarada na API:", license_metadata)
if not license_metadata:
    print("ATENÇÃO: API não expôs licença estruturada; manter ressalva documental.")

selected_names = required_names + [n for n in optional_names if n in file_index]
recovery_inventory = []

for index, name in enumerate(selected_names, start=1):
    info = file_index[name]
    expected_size = int(info.get("size", -1))
    checksum = info.get("checksum", "")
    if not checksum.startswith("md5:") or expected_size < 0:
        raise RuntimeError(f"Metadados de tamanho/checksum inválidos para {name}: {info}")
    expected_md5 = checksum.split(":", 1)[1].lower()
    target = SOURCE_ROOT / name
    target.parent.mkdir(parents=True, exist_ok=True)

    valid_existing = (
        target.is_file()
        and target.stat().st_size == expected_size
        and digest_file(target, "md5") == expected_md5
    )
    if valid_existing:
        print(f"[{index}/{len(selected_names)}] VALIDADO | {name}")
    else:
        if target.exists():
            stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
            backup = target.with_name(target.name + f".invalid-{stamp}")
            target.replace(backup)
            print(f"BACKUP DE ARQUIVO INVÁLIDO | {backup.name}")

        links = info.get("links", {})
        download_url = links.get("download") or links.get("content") or links.get("self")
        if not download_url:
            raise RuntimeError(f"URL de download ausente para {name}")

        temp = target.with_name(target.name + ".part")
        md5 = hashlib.md5()
        sha256 = hashlib.sha256()
        size = 0
        print(f"[{index}/{len(selected_names)}] BAIXANDO | {name} | {expected_size / 1e9:.3f} GB")
        with requests.get(download_url, stream=True, timeout=(30, 240)) as download:
            download.raise_for_status()
            with open(temp, "wb") as output:
                for chunk in download.iter_content(chunk_size=8 * 1024 * 1024):
                    if not chunk:
                        continue
                    output.write(chunk)
                    md5.update(chunk)
                    sha256.update(chunk)
                    size += len(chunk)
        if size != expected_size or md5.hexdigest().lower() != expected_md5:
            temp.unlink(missing_ok=True)
            raise RuntimeError(
                f"Integridade divergente em {name}: size={size}/{expected_size}, "
                f"md5={md5.hexdigest()}/{expected_md5}"
            )
        temp.replace(target)

    local_sha256 = digest_file(target, "sha256")
    if name in EXPECTED_RASTER_SHA256 and local_sha256 != EXPECTED_RASTER_SHA256[name]:
        raise RuntimeError(f"SHA-256 histórico divergente para {name}: {local_sha256}")
    recovery_inventory.append({
        "name": name,
        "relative_path": name,
        "size_bytes": target.stat().st_size,
        "zenodo_md5": expected_md5,
        "local_sha256": local_sha256,
        "download_url": download_url if not valid_existing else None,
        "status": "VERIFIED",
    })

source_manifest = {
    "dataset_id": "maize_seedling_orthomosaics_zenodo_15235602",
    "record_id": ZENODO_RECORD_ID,
    "doi": record.get("doi", "10.5281/zenodo.15235602"),
    "record_version": metadata.get("version"),
    "title": metadata.get("title"),
    "license_metadata": license_metadata,
    "license_caveat": "Preservar a ressalva: direitos de terceiros do depositante não foram verificados independentemente.",
    "retrieved_at_utc": datetime.now(timezone.utc).isoformat(),
    "source_root": str(SOURCE_ROOT),
    "files": recovery_inventory,
    "result": "PASS",
}
atomic_json(ARTIFACT_ROOT / "manifests" / "phase_2_5_source_manifest.json", source_manifest)
print("\nSOURCE RECOVERY: PASS")
print("Arquivos recuperados/validados:", len(recovery_inventory))
print("Dados persistidos em:", SOURCE_ROOT)
print("Manifesto:", ARTIFACT_ROOT / "manifests" / "phase_2_5_source_manifest.json")

## 5. Caracterização do ambiente e fontes

Com os arquivos persistidos e verificados, a próxima etapa será reabrir os GeoTIFFs por janela, confirmar dimensões/CRS/transform/bandas e ler os GT Shapefiles. Esses dados serão ligados ao manifesto existente. A implementação espacial e a materialização do dataset ficam em módulos testáveis no repositório, não neste notebook.

## 6. Roadmap operacional

| Marco | Objetivo | Estado |
|---|---|---|
| Bootstrap | Drive, branch, ambiente e identidade de execução | Implementado neste notebook |
| Recovery | Recuperar fontes e validar checksums | Implementado neste notebook |
| L3 | Seleção espacial, deduplicação e splits | Próximo |
| L4 | Materialização RGB/YOLO e conservação de GT | Pendente |
| L5 | Manifesto, integridade e leakage espacial | Pendente |
| L6 | Smoke test e treinamento controlado | Bloqueado pelos gates de dados |
| L7 | Avaliação e relatório reproduzível | Pendente |

Depois da primeira prova de percepção, a plataforma evolui para Sensor Intelligence, Environmental Intelligence, Multimodal Fusion, Temporal Intelligence, Agricultural Context, Predictive/Anomaly Intelligence e Decision Intelligence.

## 7. Limites deste milestone

- Este notebook ainda não declara L3, L4 ou L5 aprovadas.
- Nenhum treinamento ou inferência foi executado.
- A recuperação do dataset não equivale à aprovação jurídica de todos os direitos de uso.
- Os resultados agronômicos exigem validação independente em campo.
- O próximo incremento deverá reutilizar os contratos e módulos já existentes, sem duplicar Data Engine, percepção ou avaliação.

In [3]:
from pathlib import Path
import subprocess

CANDIDATES = [
    Path("/content/agri-intelligence-platform-v1"),
    Path("/content/drive/MyDrive/ArqTechLabs/Agri-Intelligence/agri-intelligence-platform-v1"),
    Path("/content/drive/MyDrive/ArqTechLabs/Agri-Intelligence"),
]

def find_repo():
    for candidate in CANDIDATES:
        if (candidate / ".git").exists():
            return candidate

    content = Path("/content")
    for git_dir in content.glob("*/.git"):
        if git_dir.is_dir() or git_dir.is_file():
            if git_dir.parent.name == "agri-intelligence-platform-v1":
                return git_dir.parent

    return None

repo = find_repo()

print("=" * 65)
print("AGRI-INTELLIGENCE — LOCALIZAÇÃO DO REPOSITÓRIO")
print("=" * 65)

if repo is None:
    print("STATUS: REPOSITÓRIO NÃO LOCALIZADO")
    print("\nPastas disponíveis em /content:")
    for p in Path("/content").iterdir():
        print(" -", p)
else:
    print("REPOSITÓRIO:", repo)

    def git(args):
        return subprocess.run(
            ["git", *args],
            cwd=repo,
            capture_output=True,
            text=True,
            check=True
        )

    print("BRANCH:", git(["branch", "--show-current"]).stdout.strip())
    print("HEAD:", git(["rev-parse", "HEAD"]).stdout.strip())
    print("WORKTREE:")
    print(git(["status", "--short"]).stdout or "CLEAN")

    print("\nCHECKPOINT:")
    print(
        git([
            "log", "-1",
            "--format=%H%n%s"
        ]).stdout.strip()
    )

print("=" * 65)

AGRI-INTELLIGENCE — LOCALIZAÇÃO DO REPOSITÓRIO
STATUS: REPOSITÓRIO NÃO LOCALIZADO

Pastas disponíveis em /content:
 - /content/.config
 - /content/sample_data


In [4]:

from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

ROOT = Path("/content/drive/MyDrive/ArqTechLabs/Agri-Intelligence")

print("=" * 65)
print("AGRI-INTELLIGENCE — RECUPERAÇÃO DO CHECKPOINT")
print("=" * 65)

if not ROOT.exists():
    print("ERRO: pasta principal não encontrada:", ROOT)
else:
    print("DRIVE ROOT:", ROOT)
    print("\nCONTEÚDO PRINCIPAL:")

    for p in sorted(ROOT.iterdir()):
        tipo = "DIR " if p.is_dir() else "FILE"
        print(f"[{tipo}] {p.name}")

    backup = ROOT / "artifacts" / "backups" / "notebook_checkpoints"

    print("\nNOTEBOOK CHECKPOINTS:")
    if backup.exists():
        for p in sorted(backup.glob("*")):
            print(p.name, "|", p.stat().st_size, "bytes")
    else:
        print("Pasta não encontrada:", backup)

    manifests = ROOT / "artifacts" / "manifests"
    print("\nMANIFESTS:")
    if manifests.exists():
        for p in sorted(manifests.glob("*")):
            print(p.name)
    else:
        print("Pasta não encontrada:", manifests)

print("=" * 65)

Mounted at /content/drive
AGRI-INTELLIGENCE — RECUPERAÇÃO DO CHECKPOINT
DRIVE ROOT: /content/drive/MyDrive/ArqTechLabs/Agri-Intelligence

CONTEÚDO PRINCIPAL:
[DIR ] artifacts
[DIR ] data
[DIR ] logs
[DIR ] state

NOTEBOOK CHECKPOINTS:
Agri_Intelligence_Platform_v1_20261110T024531Z.ipynb | 313379 bytes

MANIFESTS:
l2_global_quota_fix_report.json
l2_median_split_fix.json
l3_final_candidate_selection.json
l3_optimized_selection.json
l3_pairing_overlap_audit.json
l3_selected_blocks.json
l3_spatial_preflight.json
l3_stratified_selection.json
l3_tile_gt_audit.json
phase_2_5_l2_engineering_review.json
phase_2_5_l2_implementation.json
phase_2_5_source_manifest.json


In [5]:

from pathlib import Path

ROOT = Path("/content/drive/MyDrive/ArqTechLabs/Agri-Intelligence")

print("=" * 70)
print("AGRI-INTELLIGENCE — INVENTÁRIO DE RECUPERAÇÃO")
print("=" * 70)

for folder_name in ["state", "logs", "artifacts", "data"]:
    folder = ROOT / folder_name
    print(f"\n[{folder_name.upper()}]")

    if not folder.exists():
        print("Pasta ausente")
        continue

    files = sorted(
        folder.rglob("*"),
        key=lambda p: (len(p.parts), str(p).lower())
    )

    if not files:
        print("Vazia")
        continue

    for p in files:
        if p.is_file():
            print(f"FILE | {p.relative_to(ROOT)} | {p.stat().st_size:,} bytes")
        elif len(p.relative_to(ROOT).parts) <= 3:
            print(f"DIR  | {p.relative_to(ROOT)}/")

print("\n" + "=" * 70)
print("ARQUIVOS DE RECUPERAÇÃO ENCONTRADOS")
print("=" * 70)

patterns = ["*.bundle", "*.zip", "*.tar", "*.tar.gz", "*.patch", "*.json", "*.ipynb"]

for pattern in patterns:
    matches = list(ROOT.rglob(pattern))
    print(f"\n{pattern}: {len(matches)}")
    for p in matches[:30]:
        print(" -", p.relative_to(ROOT))

print("\n" + "=" * 70)

AGRI-INTELLIGENCE — INVENTÁRIO DE RECUPERAÇÃO

[STATE]
FILE | state/phase_2_5_bundle_recovery.json | 311 bytes
FILE | state/runtime_environment.json | 438 bytes

[LOGS]
Vazia

[ARTIFACTS]
DIR  | artifacts/backups/
DIR  | artifacts/manifests/
DIR  | artifacts/backups/l2_correction_20261110_022205/
DIR  | artifacts/backups/l2_global_quota_fix_20261110_022717/
DIR  | artifacts/backups/l2_median_fix_20261110_022402/
DIR  | artifacts/backups/notebook_checkpoints/
FILE | artifacts/manifests/l2_global_quota_fix_report.json | 694 bytes
FILE | artifacts/manifests/l2_median_split_fix.json | 1,042 bytes
FILE | artifacts/manifests/l3_final_candidate_selection.json | 81,608 bytes
FILE | artifacts/manifests/l3_optimized_selection.json | 100,034 bytes
FILE | artifacts/manifests/l3_pairing_overlap_audit.json | 3,675 bytes
FILE | artifacts/manifests/l3_selected_blocks.json | 54,662 bytes
FILE | artifacts/manifests/l3_spatial_preflight.json | 5,735 bytes
FILE | artifacts/manifests/l3_stratified_selectio

In [6]:

import json
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/ArqTechLabs/Agri-Intelligence")

files = [
    ROOT / "state/phase_2_5_bundle_recovery.json",
    ROOT / "state/runtime_environment.json",
]

for path in files:
    print("\n" + "=" * 70)
    print(path.name)
    print("=" * 70)

    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    print(json.dumps(data, indent=2, ensure_ascii=False))

for folder in [
    ROOT / "data/interim/phase_2_5",
    ROOT / "data/processed/phase_2_5",
]:
    print("\n" + "=" * 70)
    print("DIRETÓRIO:", folder)
    print("=" * 70)

    if folder.exists():
        items = list(folder.rglob("*"))
        print("Itens:", len(items))
        for p in items[:30]:
            if p.is_file():
                print(p.relative_to(ROOT), "|", p.stat().st_size, "bytes")
            else:
                print(p.relative_to(ROOT), "/")


phase_2_5_bundle_recovery.json
{
  "bundle_path": "/content/drive/MyDrive/agri_phase25/phase25_protocol.bundle",
  "source_ref": "refs/heads/feat/phase-2-5-first-real-experiment",
  "imported_ref": "refs/remotes/backup/phase25",
  "commit": "0cb46c62191c06125c20f0636e66bdf4fa74bfbe",
  "verified_at_utc": "2026-11-10T01:45:27.657747+00:00"
}

runtime_environment.json
{
  "python": "3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]",
  "platform": "Linux-6.6.159+-x86_64-with-glibc2.39",
  "modules": {
    "numpy": true,
    "pandas": true,
    "rasterio": true,
    "geopandas": true,
    "shapely": true,
    "requests": true,
    "PIL": true
  },
  "branch": "feat/platform-notebook-v1",
  "commit": "d8611bda5765449a9b79fb88d824476172cf89f2",
  "checked_at_utc": "2026-11-10T01:45:27.696340+00:00"
}

DIRETÓRIO: /content/drive/MyDrive/ArqTechLabs/Agri-Intelligence/data/interim/phase_2_5
Itens: 0

DIRETÓRIO: /content/drive/MyDrive/ArqTechLabs/Agri-Intelligence/data/processed/phase_2_5
Iten

In [7]:

import subprocess
from pathlib import Path

REPO_URL = "https://github.com/edu-moraess/agri-intelligence.git"
BRANCH = "feat/platform-notebook-v1"
DEST = Path("/content/agri-intelligence-platform-v1")

def run(cmd, cwd=None):
    p = subprocess.run(
        cmd, cwd=cwd, text=True,
        capture_output=True
    )
    if p.returncode != 0:
        raise RuntimeError(
            f"FALHA: {' '.join(map(str, cmd))}\n{p.stderr}"
        )
    return p.stdout.strip()

print("RECUPERANDO REPOSITÓRIO...")

if DEST.exists():
    raise RuntimeError(
        f"O destino já existe: {DEST}. Não vou sobrescrever."
    )

run([
    "git", "clone",
    "--single-branch",
    "--branch", BRANCH,
    REPO_URL,
    str(DEST)
])

branch = run(["git", "branch", "--show-current"], DEST)
head = run(["git", "rev-parse", "HEAD"], DEST)
remote = run(["git", "remote", "get-url", "origin"], DEST)
status = run(["git", "status", "--short"], DEST)

print("\n" + "=" * 65)
print("RECUPERAÇÃO DO REPOSITÓRIO")
print("=" * 65)
print("REMOTE:", remote)
print("BRANCH:", branch)
print("HEAD:", head)
print("WORKTREE:", status or "CLEAN")
print("CHECKPOINT LOCAL ESPERADO: fc6452b9b177439825175d9ed4d8d7f7206c401a")
print("CHECKPOINT PRESENTE NO CLONE:",
      "SIM" if run([
          "git", "cat-file", "-e",
          "fc6452b9b177439825175d9ed4d8d7f7206c401a^{commit}"
      ], DEST) == "" else "NÃO")
print("=" * 65)

RECUPERANDO REPOSITÓRIO...

RECUPERAÇÃO DO REPOSITÓRIO
REMOTE: https://github.com/edu-moraess/agri-intelligence.git
BRANCH: feat/platform-notebook-v1
HEAD: d8611bda5765449a9b79fb88d824476172cf89f2
WORKTREE: CLEAN
CHECKPOINT LOCAL ESPERADO: fc6452b9b177439825175d9ed4d8d7f7206c401a


RuntimeError: FALHA: git cat-file -e fc6452b9b177439825175d9ed4d8d7f7206c401a^{commit}
fatal: Not a valid object name fc6452b9b177439825175d9ed4d8d7f7206c401a^{commit}


In [8]:

from pathlib import Path
import shutil
import subprocess
import json

REPO = Path("/content/agri-intelligence-platform-v1")
DRIVE = Path("/content/drive/MyDrive/ArqTechLabs/Agri-Intelligence")

def git(args):
    p = subprocess.run(
        ["git", *args], cwd=REPO,
        capture_output=True, text=True
    )
    if p.returncode:
        raise RuntimeError(p.stderr)
    return p.stdout.strip()

assert REPO.exists() and (REPO / ".git").exists()
assert git(["branch", "--show-current"]) == "feat/platform-notebook-v1"

print("=" * 65)
print("RECONSTRUÇÃO DO CHECKPOINT — PHASE 2.5")
print("=" * 65)

# 1. Restaurar manifests experimentais
src = DRIVE / "artifacts/manifests"
dst = REPO / "docs/experiments/phase_2_5/manifests"
dst.mkdir(parents=True, exist_ok=True)

manifest_count = 0
for f in src.glob("*.json"):
    shutil.copy2(f, dst / f.name)
    manifest_count += 1

# 2. Restaurar implementação L2 mais recente
backup = DRIVE / "artifacts/backups/l2_global_quota_fix_20261110_022717"

assert (backup / "spatial_split.py").exists()
assert (backup / "test_spatial_split.py").exists()

code_dst = REPO / "src/agri_intelligence/data"
test_dst = REPO / "tests/unit"
code_dst.mkdir(parents=True, exist_ok=True)
test_dst.mkdir(parents=True, exist_ok=True)

shutil.copy2(backup / "spatial_split.py", code_dst / "spatial_split.py")
shutil.copy2(backup / "test_spatial_split.py", test_dst / "test_spatial_split.py")

# 3. Preservar notebook capturado da sessão anterior
notebook_src = DRIVE / "artifacts/backups/notebook_checkpoints/Agri_Intelligence_Platform_v1_20261110T024531Z.ipynb"
notebook_dst = REPO / "notebooks/Agri_Intelligence_Platform_v1.ipynb"

assert notebook_src.exists()
notebook_dst.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(notebook_src, notebook_dst)

# 4. Preservar protocolo L2 disponível no backup
protocol_src = DRIVE / "artifacts/backups/l2_correction_20261110_022205/PHASE_2_5_PROTOCOL.md"
protocol_dst = REPO / "docs/experiments/phase_2_5/PHASE_2_5_PROTOCOL_L2_BACKUP.md"

if protocol_src.exists():
    protocol_dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(protocol_src, protocol_dst)

# 5. Criar registro explícito de recuperação
recovery = {
    "status": "RECOVERED_FROM_DRIVE_ARTIFACTS",
    "previous_checkpoint_commit": "fc6452b9b177439825175d9ed4d8d7f7206c401a",
    "previous_checkpoint_commit_available": False,
    "recovered_base_commit": git(["rev-parse", "HEAD"]),
    "branch": git(["branch", "--show-current"]),
    "manifests_restored": manifest_count,
    "notebook_restored": str(notebook_dst),
    "l2_source_restored": str(code_dst / "spatial_split.py"),
    "l2_test_restored": str(test_dst / "test_spatial_split.py"),
    "data_source": str(DRIVE / "data/raw/zenodo_15235602"),
    "training_started": False
}

recovery_path = REPO / "docs/experiments/phase_2_5/RECOVERY_CHECKPOINT.json"
recovery_path.parent.mkdir(parents=True, exist_ok=True)
recovery_path.write_text(
    json.dumps(recovery, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("\nARQUIVOS RESTAURADOS")
print("Manifests:", manifest_count)
print("Notebook: OK")
print("L2 implementation: OK")
print("L2 test: OK")
print("Recovery record: OK")

print("\nGIT STATUS")
print(git(["status", "--short"]))

print("\nBASE:", git(["rev-parse", "--short", "HEAD"]))
print("BRANCH:", git(["branch", "--show-current"]))
print("\nRECONSTRUÇÃO DOS ARTEFATOS: CONCLUÍDA")
print("Nenhum treinamento ou push executado.")

RECONSTRUÇÃO DO CHECKPOINT — PHASE 2.5

ARQUIVOS RESTAURADOS
Manifests: 12
Notebook: OK
L2 implementation: OK
L2 test: OK
Recovery record: OK

GIT STATUS
M notebooks/Agri_Intelligence_Platform_v1.ipynb
?? docs/experiments/
?? src/agri_intelligence/data/spatial_split.py
?? tests/unit/

BASE: d8611bd
BRANCH: feat/platform-notebook-v1

RECONSTRUÇÃO DOS ARTEFATOS: CONCLUÍDA
Nenhum treinamento ou push executado.


In [9]:

from pathlib import Path
import subprocess

REPO = Path("/content/agri-intelligence-platform-v1")

required = [
    "src/agri_intelligence/data/spatial_split.py",
    "src/agri_intelligence/data/gt_conversion.py",
    "tests/unit/test_spatial_split.py",
    "tests/unit/test_gt_conversion.py",
    "tests/unit/test_l2_integration.py",
    "docs/PHASE_2_5_PROTOCOL.md",
    "notebooks/Agri_Intelligence_Platform_v1.ipynb",
    "docs/experiments/phase_2_5/RECOVERY_CHECKPOINT.json",
]

print("AGRI-INTELLIGENCE — RECOVERY GATE")
print("=" * 65)

for item in required:
    p = REPO / item
    print(("PASS" if p.is_file() else "MISSING"), item)

print("\nMANIFESTS:", len(list(
    (REPO / "docs/experiments/phase_2_5/manifests").glob("*.json")
)))

p = subprocess.run(
    ["git", "status", "--short"],
    cwd=REPO, capture_output=True, text=True
)
print("\nGIT STATUS:\n", p.stdout or "CLEAN")

print("=" * 65)

AGRI-INTELLIGENCE — RECOVERY GATE
PASS src/agri_intelligence/data/spatial_split.py
MISSING src/agri_intelligence/data/gt_conversion.py
PASS tests/unit/test_spatial_split.py
MISSING tests/unit/test_gt_conversion.py
MISSING tests/unit/test_l2_integration.py
MISSING docs/PHASE_2_5_PROTOCOL.md
PASS notebooks/Agri_Intelligence_Platform_v1.ipynb
PASS docs/experiments/phase_2_5/RECOVERY_CHECKPOINT.json

MANIFESTS: 12

GIT STATUS:
  M notebooks/Agri_Intelligence_Platform_v1.ipynb
?? docs/experiments/
?? src/agri_intelligence/data/spatial_split.py
?? tests/unit/



In [10]:

import subprocess
from pathlib import Path

REPO = Path("/content/agri-intelligence-platform-v1")

def git(args, check=True):
    p = subprocess.run(
        ["git", *args],
        cwd=REPO,
        capture_output=True,
        text=True
    )
    if check and p.returncode:
        raise RuntimeError(p.stderr)
    return p.stdout.strip()

print("=" * 70)
print("BUSCA DE ARQUIVOS AUSENTES NO HISTÓRICO REMOTO")
print("=" * 70)

print("\n1. Atualizando referências remotas...")
print(git(["fetch", "origin", "--prune"]))

print("\n2. Branches remotas:")
print(git(["branch", "-r"]))

targets = [
    "src/agri_intelligence/data/gt_conversion.py",
    "tests/unit/test_gt_conversion.py",
    "tests/unit/test_l2_integration.py",
    "docs/PHASE_2_5_PROTOCOL.md",
]

print("\n3. Histórico dos arquivos:")
for path in targets:
    print(f"\n--- {path} ---")
    result = git(
        ["log", "--all", "--format=%H | %ad | %s",
         "--date=short", "--", path],
        check=False
    )
    print(result if result else "NÃO ENCONTRADO NO HISTÓRICO")

print("\n4. Verificando a branch de recuperação anterior:")
ref = "origin/feat/phase-2-5-first-real-experiment"
exists = subprocess.run(
    ["git", "rev-parse", "--verify", ref],
    cwd=REPO, capture_output=True, text=True
).returncode == 0

if exists:
    print("REF:", ref)
    print("HEAD:", git(["rev-parse", ref]))
    for path in targets:
        result = subprocess.run(
            ["git", "cat-file", "-e", f"{ref}:{path}"],
            cwd=REPO, capture_output=True
        )
        print(("FOUND " if result.returncode == 0 else "MISSING "), path)
else:
    print("Branch anterior não encontrada no remoto.")

print("\n" + "=" * 70)
print("BUSCA CONCLUÍDA — NENHUM ARQUIVO FOI ALTERADO")
print("=" * 70)

BUSCA DE ARQUIVOS AUSENTES NO HISTÓRICO REMOTO

1. Atualizando referências remotas...


2. Branches remotas:
origin/feat/platform-notebook-v1

3. Histórico dos arquivos:

--- src/agri_intelligence/data/gt_conversion.py ---
NÃO ENCONTRADO NO HISTÓRICO

--- tests/unit/test_gt_conversion.py ---
NÃO ENCONTRADO NO HISTÓRICO

--- tests/unit/test_l2_integration.py ---
NÃO ENCONTRADO NO HISTÓRICO

--- docs/PHASE_2_5_PROTOCOL.md ---
NÃO ENCONTRADO NO HISTÓRICO

4. Verificando a branch de recuperação anterior:
Branch anterior não encontrada no remoto.

BUSCA CONCLUÍDA — NENHUM ARQUIVO FOI ALTERADO


In [11]:

import json
import re
import hashlib
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/ArqTechLabs/Agri-Intelligence")
NOTEBOOK = ROOT / "artifacts/backups/notebook_checkpoints/Agri_Intelligence_Platform_v1_20261110T024531Z.ipynb"

assert NOTEBOOK.exists(), f"Notebook ausente: {NOTEBOOK}"

with open(NOTEBOOK, "r", encoding="utf-8") as f:
    nb = json.load(f)

terms = [
    "gt_conversion",
    "assign_spatial_splits",
    "spatial_split",
    "Polygon",
    "MultiPolygon",
    "ImageFrame",
    "TileRecord",
    "DatasetManifest",
    "PHASE_2_5_PROTOCOL",
    "PARTIAL_LT_50",
    "OUTSIDE_SELECTED_TILES",
    "audit_split_integrity",
]

matches = []

for i, cell in enumerate(nb.get("cells", [])):
    if cell.get("cell_type") != "code":
        continue

    source = "".join(cell.get("source", []))
    found = [term for term in terms if term.lower() in source.lower()]

    if found:
        matches.append({
            "cell": i,
            "terms": found,
            "source": source
        })

out = ROOT / "artifacts/backups/notebook_recovery_candidates"
out.mkdir(parents=True, exist_ok=True)

report = []

for item in matches:
    filename = f"cell_{item['cell']:02d}.py"
    path = out / filename
    path.write_text(item["source"], encoding="utf-8")

    report.append({
        "cell": item["cell"],
        "file": filename,
        "terms": item["terms"],
        "lines": len(item["source"].splitlines()),
        "sha256": hashlib.sha256(
            item["source"].encode()
        ).hexdigest()
    })

report_path = out / "recovery_index.json"
report_path.write_text(
    json.dumps(report, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("=" * 65)
print("NOTEBOOK — RECUPERAÇÃO DE CÓDIGO")
print("=" * 65)
print("Total de células:", len(nb.get("cells", [])))
print("Células relevantes:", len(matches))

for item in report:
    print(
        f"\nCélula {item['cell']} | {item['lines']} linhas"
        f"\nTermos: {', '.join(item['terms'])}"
        f"\nArquivo: {item['file']}"
    )

print("\nÍNDICE:", report_path)
print("CANDIDATOS:", out)
print("=" * 65)

NOTEBOOK — RECUPERAÇÃO DE CÓDIGO
Total de células: 35
Células relevantes: 16

Célula 16 | 52 linhas
Termos: gt_conversion, spatial_split, PHASE_2_5_PROTOCOL
Arquivo: cell_16.py

Célula 17 | 111 linhas
Termos: PHASE_2_5_PROTOCOL
Arquivo: cell_17.py

Célula 18 | 732 linhas
Termos: gt_conversion, assign_spatial_splits, spatial_split, Polygon, MultiPolygon, PHASE_2_5_PROTOCOL, PARTIAL_LT_50, OUTSIDE_SELECTED_TILES, audit_split_integrity
Arquivo: cell_18.py

Célula 19 | 55 linhas
Termos: gt_conversion, spatial_split
Arquivo: cell_19.py

Célula 20 | 452 linhas
Termos: gt_conversion, spatial_split, Polygon, MultiPolygon, PHASE_2_5_PROTOCOL, PARTIAL_LT_50, OUTSIDE_SELECTED_TILES
Arquivo: cell_20.py

Célula 21 | 526 linhas
Termos: gt_conversion, spatial_split, Polygon, MultiPolygon, ImageFrame, TileRecord, DatasetManifest, PHASE_2_5_PROTOCOL, PARTIAL_LT_50, OUTSIDE_SELECTED_TILES
Arquivo: cell_21.py

Célula 22 | 29 linhas
Termos: spatial_split
Arquivo: cell_22.py

Célula 23 | 246 linhas
Termos:

In [1]:

import json, re, hashlib
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/ArqTechLabs/Agri-Intelligence")
NB_PATH = ROOT / "artifacts/backups/notebook_checkpoints/Agri_Intelligence_Platform_v1_20261110T024531Z.ipynb"
OUT = ROOT / "artifacts/backups/notebook_recovery_candidates"

nb = json.loads(NB_PATH.read_text(encoding="utf-8"))
records = []

print("=" * 75)
print("RECUPERAÇÃO ESTRUTURAL — IMPLEMENTAÇÃO L2 / GT CONVERSION")
print("=" * 75)

for i, cell in enumerate(nb["cells"]):
    if cell.get("cell_type") != "code":
        continue

    source = "".join(cell.get("source", []))
    if not any(x in source.lower() for x in [
        "gt_conversion", "spatial_split", "assign_spatial_splits",
        "polygon", "multpolygon", "imagerecord", "tilerecord",
        "datasetmanifest"
    ]):
        continue

    definitions = re.findall(
        r"(?m)^\s*(?:async\s+)?(?:def|class)\s+([A-Za-z_]\w*)",
        source
    )
    imports = re.findall(
        r"(?m)^\s*(?:from\s+[\w.]+\s+)?import\s+([^\n]+)",
        source
    )
    headings = [
        line.strip()[:150]
        for line in source.splitlines()
        if line.strip().startswith(("#", '"""', "'''"))
    ][:8]

    digest = hashlib.sha256(source.encode()).hexdigest()

    records.append({
        "cell": i,
        "lines": len(source.splitlines()),
        "sha256": digest,
        "definitions": definitions,
        "imports": imports[:12],
        "headings": headings
    })

    print(f"\nCELL {i} | {len(source.splitlines())} linhas")
    print("SHA256:", digest)
    print("DEFINIÇÕES:", ", ".join(definitions) or "Nenhuma")
    if headings:
        print("TÍTULOS/COMENTÁRIOS:")
        for h in headings:
            print(" ", h)

index = OUT / "structural_recovery_index.json"
index.write_text(
    json.dumps(records, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("\n" + "=" * 75)
print("ÍNDICE ESTRUTURAL SALVO:", index)
print("CÉLULAS ANALISADAS:", len(records))
print("=" * 75)

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/ArqTechLabs/Agri-Intelligence/artifacts/backups/notebook_checkpoints/Agri_Intelligence_Platform_v1_20261110T024531Z.ipynb'

In [2]:

from google.colab import drive
from pathlib import Path
import json
import re
import hashlib

# 1. Recuperar acesso ao Drive
drive.mount("/content/drive")

MYDRIVE = Path("/content/drive/MyDrive")
FILENAME = "Agri_Intelligence_Platform_v1_20261110T024531Z.ipynb"

# 2. Localizar o notebook, mesmo se estiver em outra pasta
matches = list(MYDRIVE.rglob(FILENAME))

if not matches:
    raise FileNotFoundError(
        f"Notebook {FILENAME} não encontrado no Google Drive. "
        "Nenhum arquivo foi alterado."
    )

NB_PATH = matches[0]
print("NOTEBOOK LOCALIZADO:", NB_PATH)

# 3. Recuperar células e identificar implementações
nb = json.loads(NB_PATH.read_text(encoding="utf-8"))

OUT = NB_PATH.parent.parent / "notebook_recovery_candidates"
OUT.mkdir(parents=True, exist_ok=True)

terms = [
    "gt_conversion", "spatial_split", "assign_spatial_splits",
    "Polygon", "MultiPolygon", "ImageFrame", "TileRecord",
    "DatasetManifest", "audit_split_integrity"
]

records = []

print("\n" + "=" * 70)
print("RECUPERAÇÃO ESTRUTURAL DO NOTEBOOK")
print("=" * 70)
print("Total de células:", len(nb.get("cells", [])))

for i, cell in enumerate(nb.get("cells", [])):
    if cell.get("cell_type") != "code":
        continue

    source = "".join(cell.get("source", []))
    if not any(t.lower() in source.lower() for t in terms):
        continue

    # Preservar código recuperado
    filename = f"cell_{i:02d}.py"
    (OUT / filename).write_text(source, encoding="utf-8")

    definitions = re.findall(
        r"(?m)^\s*(?:async\s+)?(?:def|class)\s+([A-Za-z_]\w*)",
        source
    )

    record = {
        "cell": i,
        "file": filename,
        "lines": len(source.splitlines()),
        "sha256": hashlib.sha256(source.encode()).hexdigest(),
        "definitions": definitions
    }
    records.append(record)

    print(
        f"\nCélula {i} | {record['lines']} linhas"
        f"\nArquivo: {filename}"
        f"\nDefinições: {', '.join(definitions) or 'Nenhuma'}"
    )

index = OUT / "structural_recovery_index.json"
index.write_text(
    json.dumps(records, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("\n" + "=" * 70)
print("CÉLULAS RECUPERADAS:", len(records))
print("ÍNDICE:", index)
print("DIRETÓRIO:", OUT)
print("=" * 70)

Mounted at /content/drive
NOTEBOOK LOCALIZADO: /content/drive/MyDrive/ArqTechLabs/Agri-Intelligence/artifacts/backups/notebook_checkpoints/Agri_Intelligence_Platform_v1_20261110T024531Z.ipynb

RECUPERAÇÃO ESTRUTURAL DO NOTEBOOK
Total de células: 35

Célula 16 | 52 linhas
Arquivo: cell_16.py
Definições: git

Célula 18 | 732 linhas
Arquivo: cell_18.py
Definições: git, SpatialBlock, __post_init__, block_id, block_indices, block_id, largest_remainder, assign_spatial_splits, TileWindow, __post_init__, ConversionResult, _inverse_affine, convert, convert_annotation, test_global_grid_uses_floor_for_negative_coordinates, test_block_id_is_stable, test_largest_remainder_conserves_total, test_spatial_split_is_deterministic_and_conserves_blocks, test_duplicate_global_blocks_are_rejected, world_polygon, test_polygon_converts_to_tile_relative_xywh, test_north_up_affine_preserves_pixel_y_direction, test_multipolygon_is_one_instance_with_union_bounds, test_partial_exactly_fifty_percent_is_included, tes

In [3]:

import ast
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/ArqTechLabs/Agri-Intelligence")
CANDIDATES = ROOT / "artifacts/backups/notebook_recovery_candidates"

targets = {
    "cell_18.py": CANDIDATES / "cell_18.py",
    "cell_25.py": CANDIDATES / "cell_25.py",
    "cell_33.py": CANDIDATES / "cell_33.py",
}

print("=" * 75)
print("AGRI-INTELLIGENCE — MAPA DE RECUPERAÇÃO DO CÓDIGO ORIGINAL")
print("=" * 75)

for filename, path in targets.items():
    source = path.read_text(encoding="utf-8")
    tree = ast.parse(source)

    print(f"\n{'=' * 65}")
    print(filename, "|", len(source.splitlines()), "linhas")
    print("=" * 65)

    for node in tree.body:
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)):
            kind = "CLASS" if isinstance(node, ast.ClassDef) else "FUNCTION"
            print(
                f"{kind:8} | L{node.lineno:04d}-L{node.end_lineno:04d}"
                f" | {node.name}"
            )
        elif isinstance(node, (ast.Import, ast.ImportFrom)):
            print(f"IMPORT   | L{node.lineno:04d} | {ast.get_source_segment(source, node)}")
        elif isinstance(node, ast.Assign):
            names = [
                t.id for t in node.targets
                if isinstance(t, ast.Name)
            ]
            if names:
                print(f"ASSIGN   | L{node.lineno:04d} | {', '.join(names)}")

print("\n" + "=" * 75)
print("MAPEAMENTO CONCLUÍDO — NENHUM ARQUIVO DO REPOSITÓRIO ALTERADO")
print("=" * 75)

AGRI-INTELLIGENCE — MAPA DE RECUPERAÇÃO DO CÓDIGO ORIGINAL

cell_18.py | 732 linhas
IMPORT   | L0003 | from pathlib import Path
IMPORT   | L0004 | import subprocess
IMPORT   | L0005 | import sys
IMPORT   | L0006 | import textwrap
IMPORT   | L0007 | import json
IMPORT   | L0008 | from datetime import datetime, timezone
ASSIGN   | L0016 | REPO
ASSIGN   | L0017 | EXPECTED_BRANCH
ASSIGN   | L0018 | EXPECTED_HEAD
FUNCTION | L0023-L0030 | git
ASSIGN   | L0032 | branch
ASSIGN   | L0033 | head
ASSIGN   | L0034 | status
ASSIGN   | L0053 | required
ASSIGN   | L0067 | SPATIAL_SPLIT
ASSIGN   | L0220 | GT_CONVERSION
ASSIGN   | L0404 | TEST_SPATIAL
ASSIGN   | L0453 | TEST_GT
ASSIGN   | L0542 | TEST_INTEGRATION
ASSIGN   | L0591 | FILES
ASSIGN   | L0599 | protocol
IMPORT   | L0654 | import ast
ASSIGN   | L0667 | command
ASSIGN   | L0674 | result
ASSIGN   | L0690 | test_state
ASSIGN   | L0696 | report
ASSIGN   | L0709 | report_path

cell_25.py | 655 linhas
IMPORT   | L0008 | from pathlib import Path
IM

In [5]:

# ================================================================
# AGRI-INTELLIGENCE — RECUPERAÇÃO EXECUTÁVEL PHASE 2.5
# Recria o ambiente, restaura arquivos do notebook/Drive e testa.
# Não treina, não altera main e não faz push.
# ================================================================

from pathlib import Path
import os
import ast
import json
import shutil
import subprocess
import sys
import textwrap
from datetime import datetime, timezone

# ------------------------------------------------
# 1. LOCALIZAÇÃO E MONTAGEM DO DRIVE
# ------------------------------------------------

DRIVE = Path("/content/drive/MyDrive/ArqTechLabs/Agri-Intelligence")
REPO = Path("/content/agri-intelligence-platform-v1")
CANDIDATES = DRIVE / "artifacts/backups/notebook_recovery_candidates"
L2_BACKUP = DRIVE / "artifacts/backups/l2_global_quota_fix_20261110_022717"

assert DRIVE.exists(), f"Drive do projeto não encontrado: {DRIVE}"
assert CANDIDATES.exists(), f"Diretório de recuperação não encontrado: {CANDIDATES}"
assert (CANDIDATES / "cell_18.py").exists(), "cell_18.py ausente."
assert (CANDIDATES / "cell_25.py").exists(), "cell_25.py ausente."
assert (L2_BACKUP / "spatial_split.py").exists(), "Backup L2 ausente."
assert (L2_BACKUP / "test_spatial_split.py").exists(), "Teste L2 ausente."

# ------------------------------------------------
# 2. RECRIA O REPOSITÓRIO SE O RUNTIME FOI PERDIDO
# ------------------------------------------------

if not (REPO / ".git").exists():
    if REPO.exists():
        shutil.rmtree(REPO)

    print("Repositório local ausente. Clonando branch de recuperação...")
    subprocess.run(
        [
            "git", "clone",
            "--branch", "feat/platform-notebook-v1",
            "--single-branch",
            "https://github.com/edu-moraess/agri-intelligence.git",
            str(REPO)
        ],
        check=True
    )

assert (REPO / ".git").exists(), "Falha ao recriar o repositório."

os.chdir(REPO)

def git(*args):
    return subprocess.run(
        ["git", *args],
        cwd=REPO,
        text=True,
        capture_output=True,
        check=True
    ).stdout.strip()

BRANCH = git("branch", "--show-current")
HEAD = git("rev-parse", "HEAD")

assert BRANCH == "feat/platform-notebook-v1", (
    f"Branch inesperada: {BRANCH}"
)

print("REPOSITÓRIO:", REPO)
print("BRANCH:", BRANCH)
print("HEAD:", HEAD)

# ------------------------------------------------
# 3. EXTRAI ATRIBUIÇÕES DO CÓDIGO ORIGINAL DO NOTEBOOK
# ------------------------------------------------

def extract_assignment(source_path, variable_name):
    source = Path(source_path).read_text(encoding="utf-8")
    tree = ast.parse(source)

    matches = []

    for node in ast.walk(tree):
        if isinstance(node, ast.Assign):
            names = [
                target.id for target in node.targets
                if isinstance(target, ast.Name)
            ]
            if variable_name in names:
                matches.append(node.value)

        elif isinstance(node, ast.AnnAssign):
            if (
                isinstance(node.target, ast.Name)
                and node.target.id == variable_name
            ):
                matches.append(node.value)

    if not matches:
        raise KeyError(
            f"Atribuição {variable_name!r} não encontrada em {source_path}"
        )

    value_node = matches[-1]

    if isinstance(value_node, ast.Call):
        # Suporta textwrap.dedent("""...""")
        if (
            isinstance(value_node.func, ast.Attribute)
            and value_node.func.attr == "dedent"
            and value_node.args
        ):
            value = ast.literal_eval(value_node.args[0])
            value = textwrap.dedent(value)
        else:
            value = ast.literal_eval(value_node)
    else:
        value = ast.literal_eval(value_node)

    if not isinstance(value, str):
        raise TypeError(
            f"{variable_name} não resultou em texto; "
            f"tipo encontrado: {type(value).__name__}"
        )

    return value

CELL18 = CANDIDATES / "cell_18.py"
CELL25 = CANDIDATES / "cell_25.py"

GT_CONVERSION = extract_assignment(CELL18, "GT_CONVERSION")
TEST_GT = extract_assignment(CELL18, "TEST_GT")
TEST_INTEGRATION = extract_assignment(CELL18, "TEST_INTEGRATION")
PROTOCOL = extract_assignment(CELL18, "protocol")
GLOBAL_QUOTA_TEST = extract_assignment(CELL25, "test_text")

# ------------------------------------------------
# 4. BACKUP DOS ARQUIVOS ANTES DA RESTAURAÇÃO
# ------------------------------------------------

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
recovery_backup = DRIVE / "artifacts/backups" / f"recovery_execution_{stamp}"
recovery_backup.mkdir(parents=True, exist_ok=True)

TARGETS = [
    REPO / "src/agri_intelligence/data/spatial_split.py",
    REPO / "tests/unit/test_spatial_split.py",
    REPO / "src/agri_intelligence/data/gt_conversion.py",
    REPO / "tests/unit/test_gt_conversion.py",
    REPO / "tests/unit/test_l2_integration.py",
    REPO / "docs/PHASE_2_5_PROTOCOL.md",
    REPO / "notebooks/Agri_Intelligence_Platform_v1.ipynb",
]

for target in TARGETS:
    if target.exists():
        relative = target.relative_to(REPO)
        destination = recovery_backup / relative
        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(target, destination)

print("Backup pré-restauração:", recovery_backup)

# ------------------------------------------------
# 5. RESTAURA OS ARQUIVOS
# ------------------------------------------------

def write_text(relative_path, content):
    target = REPO / relative_path
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(content.rstrip() + "\n", encoding="utf-8")
    print("RESTORED:", relative_path)

shutil.copy2(
    L2_BACKUP / "spatial_split.py",
    REPO / "src/agri_intelligence/data/spatial_split.py"
)
print("RESTORED: src/agri_intelligence/data/spatial_split.py")

shutil.copy2(
    L2_BACKUP / "test_spatial_split.py",
    REPO / "tests/unit/test_spatial_split.py"
)
print("RESTORED: tests/unit/test_spatial_split.py")

write_text(
    "src/agri_intelligence/data/gt_conversion.py",
    GT_CONVERSION
)

write_text(
    "tests/unit/test_gt_conversion.py",
    TEST_GT
)

write_text(
    "tests/unit/test_l2_integration.py",
    TEST_INTEGRATION
)

write_text(
    "docs/PHASE_2_5_PROTOCOL.md",
    PROTOCOL
)

# Incorpora o teste de quotas globais, sem duplicá-lo.
spatial_test_path = REPO / "tests/unit/test_spatial_split.py"
spatial_test = spatial_test_path.read_text(encoding="utf-8")

if "test_global_quotas_preserved_across_strata" not in spatial_test:
    with spatial_test_path.open("a", encoding="utf-8") as f:
        f.write("\n\n")
        f.write(textwrap.dedent(GLOBAL_QUOTA_TEST).strip())
        f.write("\n")
    print("RESTORED: teste de quotas globais")
else:
    print("PRESERVED: teste de quotas globais já existente")

# Restaura o notebook preservado no Drive.
notebook_backup = (
    DRIVE
    / "artifacts/backups/notebook_checkpoints/"
    / "Agri_Intelligence_Platform_v1_20261110T024531Z.ipynb"
)

if notebook_backup.exists():
    notebook_target = REPO / "notebooks/Agri_Intelligence_Platform_v1.ipynb"
    notebook_target.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(notebook_backup, notebook_target)
    print("RESTORED: notebook original")

# ------------------------------------------------
# 6. VALIDAÇÃO SINTÁTICA
# ------------------------------------------------

python_files = [
    REPO / "src/agri_intelligence/data/spatial_split.py",
    REPO / "src/agri_intelligence/data/gt_conversion.py",
    REPO / "tests/unit/test_spatial_split.py",
    REPO / "tests/unit/test_gt_conversion.py",
    REPO / "tests/unit/test_l2_integration.py",
]

for path in python_files:
    ast.parse(path.read_text(encoding="utf-8"), filename=str(path))

print("\nSTATIC AST VALIDATION: PASS")

compile_result = subprocess.run(
    [
        sys.executable, "-m", "compileall", "-q",
        "src/agri_intelligence/data",
        "tests/unit"
    ],
    cwd=REPO,
    text=True,
    capture_output=True
)

print("COMPILEALL RETURN CODE:", compile_result.returncode)

if compile_result.stdout:
    print(compile_result.stdout)
if compile_result.stderr:
    print(compile_result.stderr)

# ------------------------------------------------
# 7. TESTES L2
# ------------------------------------------------

env = os.environ.copy()
env["PYTHONPATH"] = str(REPO / "src") + os.pathsep + env.get("PYTHONPATH", "")

test_command = [
    sys.executable, "-m", "pytest", "-q",
    "tests/unit/test_spatial_split.py",
    "tests/unit/test_gt_conversion.py",
    "tests/unit/test_l2_integration.py"
]

test_result = subprocess.run(
    test_command,
    cwd=REPO,
    env=env,
    text=True,
    capture_output=True
)

print("\n" + "=" * 68)
print("L2 RECOVERY TEST RESULTS")
print("=" * 68)
print(test_result.stdout)

if test_result.stderr:
    print(test_result.stderr)

print("PYTEST RETURN CODE:", test_result.returncode)

# ------------------------------------------------
# 8. REGISTRO DE EVIDÊNCIAS
# ------------------------------------------------

manifest = {
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "branch": BRANCH,
    "base_head": HEAD,
    "repository": str(REPO),
    "drive_root": str(DRIVE),
    "recovery_backup": str(recovery_backup),
    "restored_files": [
        "src/agri_intelligence/data/spatial_split.py",
        "src/agri_intelligence/data/gt_conversion.py",
        "tests/unit/test_spatial_split.py",
        "tests/unit/test_gt_conversion.py",
        "tests/unit/test_l2_integration.py",
        "docs/PHASE_2_5_PROTOCOL.md",
        "notebooks/Agri_Intelligence_Platform_v1.ipynb"
        if notebook_backup.exists() else None
    ],
    "ast_validation": "PASS",
    "compileall_returncode": compile_result.returncode,
    "pytest_returncode": test_result.returncode,
    "pytest_output": test_result.stdout,
    "training_executed": False,
    "git_push_executed": False
}

manifest_path = (
    DRIVE / "artifacts/manifests"
    / f"recovery_execution_{stamp}.json"
)
manifest_path.parent.mkdir(parents=True, exist_ok=True)
manifest_path.write_text(
    json.dumps(manifest, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("\n" + "=" * 68)
print("RECOVERY SUMMARY")
print("=" * 68)
print("Manifest:", manifest_path)
print("AST:", "PASS")
print("COMPILEALL:", "PASS" if compile_result.returncode == 0 else "FAIL")
print("PYTEST:", "PASS" if test_result.returncode == 0 else "FAIL")
print("GIT STATUS:")
print(git("status", "--short"))

if compile_result.returncode != 0 or test_result.returncode != 0:
    raise RuntimeError(
        "Recuperação executada, mas a validação falhou. "
        "Use a saída acima para corrigir o ponto específico."
    )

print("\nL2 RECOVERY AND TESTS: PASS")

Repositório local ausente. Clonando branch de recuperação...
REPOSITÓRIO: /content/agri-intelligence-platform-v1
BRANCH: feat/platform-notebook-v1
HEAD: d8611bda5765449a9b79fb88d824476172cf89f2


ValueError: malformed node or string on line 381: <ast.Call object at 0x7acf4612aa50>

In [7]:

import ast
from pathlib import Path

def safe_literal(node):
    if isinstance(node, ast.Constant):
        return node.value

    if isinstance(node, (ast.List, ast.Tuple, ast.Set)):
        values = [safe_literal(item) for item in node.elts]

        if isinstance(node, ast.List):
            return values
        if isinstance(node, ast.Tuple):
            return tuple(values)
        return set(values)

    if isinstance(node, ast.Dict):
        return {
            safe_literal(k): safe_literal(v)
            for k, v in zip(node.keys, node.values)
        }

    if isinstance(node, ast.Call):
        if isinstance(node.func, ast.Name) and node.func.id == "set":
            if not node.args and not node.keywords:
                return set()

        if isinstance(node.func, ast.Attribute):
            method = node.func.attr

            if method == "read_text":
                raise ValueError(
                    "Leitura de arquivo detectada. "
                    "Não executar automaticamente durante "
                    "a interpretação de literais."
                )

        raise ValueError(
            f"Chamada não permitida: {ast.dump(node)}"
        )

    raise ValueError(
        f"Expressão não permitida: {type(node).__name__}"
    )

In [8]:
import ast

print("=" * 70)
print("DIAGNÓSTICO GLOBAL_QUOTA_TEST")
print("=" * 70)

for i, node in enumerate(matches):
    print(f"\nMATCH {i}")
    print(ast.dump(node, indent=2))

DIAGNÓSTICO GLOBAL_QUOTA_TEST

MATCH 0
Call(
  func=Attribute(
    value=Name(id='TEST', ctx=Load()),
    attr='read_text',
    ctx=Load()),
  keywords=[
    keyword(
      arg='encoding',
      value=Constant(value='utf-8'))])


In [9]:

import ast
from pathlib import Path

CELL25 = Path(
    "/content/drive/MyDrive/ArqTechLabs/Agri-Intelligence/"
    "artifacts/backups/notebook_recovery_candidates/cell_25.py"
)

source = CELL25.read_text(encoding="utf-8")
lines = source.splitlines()
tree = ast.parse(source)

for node in ast.walk(tree):
    if isinstance(node, ast.Assign) and any(
        isinstance(t, ast.Name) and t.id == "test_text"
        for t in node.targets
    ):
        print("ATRIBUIÇÃO test_text — LINHA", node.lineno)
        print("\nEXPRESSÃO AST:")
        print(ast.dump(node.value, indent=2))
        print("\nCONTEXTO:")
        start = max(0, node.lineno - 12)
        end = min(len(lines), node.end_lineno + 12)
        for i in range(start, end):
            print(f"{i+1:04d}: {lines[i]}")

ATRIBUIÇÃO test_text — LINHA 381

EXPRESSÃO AST:
Call(
  func=Attribute(
    value=Name(id='TEST', ctx=Load()),
    attr='read_text',
    ctx=Load()),
  keywords=[
    keyword(
      arg='encoding',
      value=Constant(value='utf-8'))])

CONTEXTO:
0370: 
0371: SRC.write_text(patched, encoding="utf-8")
0372: 
0373: print("\nPATCH:")
0374: print("assign_spatial_splits substituída: PASS")
0375: print("Validação sintática pré-gravação: PASS")
0376: 
0377: # ------------------------------------------------
0378: # 5. TESTE DE REGRESSÃO PERMANENTE
0379: # ------------------------------------------------
0380: 
0381: test_text = TEST.read_text(encoding="utf-8")
0382: 
0383: regression_name = "test_global_quotas_preserved_across_strata"
0384: 
0385: if regression_name not in test_text:
0386:     regression = r'''
0387: 
0388: def test_global_quotas_preserved_across_strata():
0389:     from collections import Counter
0390:     from agri_intelligence.data.spatial_split import (
0391:         Sp

In [10]:

import ast
from pathlib import Path

REPO = Path("/content/agri-intelligence-platform-v1")
CELL25 = Path(
    "/content/drive/MyDrive/ArqTechLabs/Agri-Intelligence/"
    "artifacts/backups/notebook_recovery_candidates/cell_25.py"
)

source = CELL25.read_text(encoding="utf-8")
tree = ast.parse(source)

def find_assignment(name):
    found = []
    for node in ast.walk(tree):
        if isinstance(node, ast.Assign):
            if any(
                isinstance(t, ast.Name) and t.id == name
                for t in node.targets
            ):
                found.append(node.value)
    if not found:
        raise RuntimeError(f"Atribuição {name} não encontrada.")
    return found[-1]

# O bloco regression é uma string literal raw no código original.
REGRESSION = ast.literal_eval(find_assignment("regression"))

test_path = REPO / "tests/unit/test_spatial_split.py"
assert test_path.exists(), f"Teste não encontrado: {test_path}"

test_text = test_path.read_text(encoding="utf-8")
regression_name = "test_global_quotas_preserved_across_strata"

if regression_name not in test_text:
    with test_path.open("a", encoding="utf-8") as f:
        f.write("\n\n")
        f.write(REGRESSION.strip())
        f.write("\n")
    print("REGRESSION TEST: RESTORED")
else:
    print("REGRESSION TEST: ALREADY PRESENT")

# Validação sintática e execução do teste específico.
import ast as _ast
_ast.parse(test_path.read_text(encoding="utf-8"))

import os
import subprocess
import sys

env = os.environ.copy()
env["PYTHONPATH"] = str(REPO / "src") + os.pathsep + env.get("PYTHONPATH", "")

result = subprocess.run(
    [
        sys.executable, "-m", "pytest", "-q",
        "tests/unit/test_spatial_split.py",
        "-k", regression_name
    ],
    cwd=REPO,
    env=env,
    text=True,
    capture_output=True
)

print(result.stdout)
if result.stderr:
    print(result.stderr)

print("RETURN CODE:", result.returncode)

if result.returncode != 0:
    raise RuntimeError("O teste de regressão não passou.")

AssertionError: Teste não encontrado: /content/agri-intelligence-platform-v1/tests/unit/test_spatial_split.py

In [11]:

from pathlib import Path
import ast
import shutil
import subprocess
import sys
import os

# Caminhos persistentes
DRIVE = Path("/content/drive/MyDrive/ArqTechLabs/Agri-Intelligence")
REPO = Path("/content/agri-intelligence-platform-v1")
L2 = DRIVE / "artifacts/backups/l2_global_quota_fix_20261110_022717"
CELL25 = DRIVE / "artifacts/backups/notebook_recovery_candidates/cell_25.py"

# 1. Confirmar origens e destino
assert (REPO / ".git").exists(), "Repositório não foi clonado."
assert (L2 / "test_spatial_split.py").exists(), "Backup do teste não encontrado."
assert CELL25.exists(), "Célula 25 não encontrada."

# 2. Restaurar o teste-base antes de qualquer alteração
test_path = REPO / "tests/unit/test_spatial_split.py"
test_path.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(L2 / "test_spatial_split.py", test_path)

print("BASE TEST RESTORED:", test_path)

# 3. Extrair somente a string literal regression da célula 25
tree = ast.parse(CELL25.read_text(encoding="utf-8"))
matches = []

for node in ast.walk(tree):
    if isinstance(node, ast.Assign) and any(
        isinstance(target, ast.Name) and target.id == "regression"
        for target in node.targets
    ):
        matches.append(node.value)

if not matches:
    raise RuntimeError("Bloco regression não encontrado na célula 25.")

REGRESSION = ast.literal_eval(matches[-1])

if not isinstance(REGRESSION, str) or not REGRESSION.strip():
    raise RuntimeError("Bloco regression vazio ou inválido.")

# 4. Aplicar regressão sem duplicar
regression_name = "test_global_quotas_preserved_across_strata"
test_text = test_path.read_text(encoding="utf-8")

if regression_name not in test_text:
    with test_path.open("a", encoding="utf-8") as f:
        f.write("\n\n" + REGRESSION.strip() + "\n")
    print("REGRESSION TEST: ADDED")
else:
    print("REGRESSION TEST: ALREADY PRESENT")

# 5. Validar sintaxe
ast.parse(test_path.read_text(encoding="utf-8"))
print("AST VALIDATION: PASS")

# 6. Executar regressão isolada
env = os.environ.copy()
env["PYTHONPATH"] = str(REPO / "src") + os.pathsep + env.get("PYTHONPATH", "")

result = subprocess.run(
    [
        sys.executable, "-m", "pytest", "-q",
        "tests/unit/test_spatial_split.py",
        "-k", regression_name
    ],
    cwd=REPO,
    env=env,
    text=True,
    capture_output=True
)

print("\nTEST OUTPUT:")
print(result.stdout)
if result.stderr:
    print(result.stderr)

print("RETURN CODE:", result.returncode)

if result.returncode != 0:
    raise RuntimeError("Regressão falhou; não avançar para a suíte completa.")

print("\nGLOBAL QUOTA REGRESSION: PASS")

BASE TEST RESTORED: /content/agri-intelligence-platform-v1/tests/unit/test_spatial_split.py
REGRESSION TEST: ADDED
AST VALIDATION: PASS

TEST OUTPUT:

==================================== ERRORS ====================================
______________ ERROR collecting tests/unit/test_spatial_split.py _______________
ImportError while importing test module '/content/agri-intelligence-platform-v1/tests/unit/test_spatial_split.py'.
Hint: make sure your test modules/packages have valid Python names.
Traceback:
/usr/lib/python3.13/importlib/__init__.py:88: in import_module
    return _bootstrap._gcd_import(name[level:], package, level)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
tests/unit/test_spatial_split.py:2: in <module>
    from agri_intelligence.data.spatial_split import (
E   ModuleNotFoundError: No module named 'agri_intelligence.data.spatial_split'
=========================== short test summary info ============================
ERROR tests/unit/test_spatial_split.py

RuntimeError: Regressão falhou; não avançar para a suíte completa.

In [12]:

from pathlib import Path
import shutil
import subprocess
import sys
import os
import ast

REPO = Path("/content/agri-intelligence-platform-v1")
DRIVE = Path("/content/drive/MyDrive/ArqTechLabs/Agri-Intelligence")
L2 = DRIVE / "artifacts/backups/l2_global_quota_fix_20261110_022717"

# 1. Restaurar o módulo que o teste precisa importar
source = L2 / "spatial_split.py"
target = REPO / "src/agri_intelligence/data/spatial_split.py"

assert (REPO / ".git").exists(), "Repositório não encontrado."
assert source.exists(), f"Backup do módulo ausente: {source}"

target.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(source, target)

# 2. Confirmar módulo e teste
test_path = REPO / "tests/unit/test_spatial_split.py"
assert target.exists(), "Módulo não foi restaurado."
assert test_path.exists(), "Arquivo de teste não encontrado."

ast.parse(target.read_text(encoding="utf-8"))
ast.parse(test_path.read_text(encoding="utf-8"))

print("MODULE RESTORED:", target)
print("MODULE SIZE:", target.stat().st_size, "bytes")

# 3. Validar importação diretamente
env = os.environ.copy()
env["PYTHONPATH"] = (
    str(REPO / "src") + os.pathsep + env.get("PYTHONPATH", "")
)

import_result = subprocess.run(
    [
        sys.executable, "-c",
        "from agri_intelligence.data.spatial_split import "
        "SpatialBlock, assign_spatial_splits, largest_remainder; "
        "print('IMPORT: PASS')"
    ],
    cwd=REPO,
    env=env,
    capture_output=True,
    text=True
)

print(import_result.stdout)
if import_result.stderr:
    print(import_result.stderr)

if import_result.returncode != 0:
    raise RuntimeError("Importação do módulo falhou.")

# 4. Executar teste de regressão
test_result = subprocess.run(
    [
        sys.executable, "-m", "pytest", "-q",
        "tests/unit/test_spatial_split.py",
        "-k", "test_global_quotas_preserved_across_strata"
    ],
    cwd=REPO,
    env=env,
    capture_output=True,
    text=True
)

print("\nREGRESSION TEST:")
print(test_result.stdout)
if test_result.stderr:
    print(test_result.stderr)

print("RETURN CODE:", test_result.returncode)

if test_result.returncode != 0:
    raise RuntimeError("Regressão ainda falhou; preservar a saída para correção.")

print("\nGLOBAL QUOTA REGRESSION: PASS")

MODULE RESTORED: /content/agri-intelligence-platform-v1/src/agri_intelligence/data/spatial_split.py
MODULE SIZE: 4632 bytes
IMPORT: PASS


REGRESSION TEST:
.                                                                        [100%]
1 passed, 6 deselected in 0.05s

RETURN CODE: 0

GLOBAL QUOTA REGRESSION: PASS


In [13]:

# ================================================================
# PHASE 2.5 — RESTAURAÇÃO DOS ARQUIVOS L2 E SUÍTE COMPLETA
# ================================================================

from pathlib import Path
import ast
import textwrap
import shutil
import subprocess
import sys
import os
import json
from datetime import datetime, timezone

REPO = Path("/content/agri-intelligence-platform-v1")
DRIVE = Path("/content/drive/MyDrive/ArqTechLabs/Agri-Intelligence")
CANDIDATES = DRIVE / "artifacts/backups/notebook_recovery_candidates"
CELL18 = CANDIDATES / "cell_18.py"
L2 = DRIVE / "artifacts/backups/l2_global_quota_fix_20261110_022717"

assert (REPO / ".git").exists(), "Repositório ausente."
assert CELL18.exists(), "Notebook recovery cell_18.py ausente."
assert (REPO / "src/agri_intelligence/data/spatial_split.py").exists()
assert (REPO / "tests/unit/test_spatial_split.py").exists()

os.chdir(REPO)

# 1. Extrair strings literais da célula original
def extract_assignment(path, variable_name):
    tree = ast.parse(path.read_text(encoding="utf-8"))
    matches = []

    for node in ast.walk(tree):
        value = None

        if isinstance(node, ast.Assign):
            if any(
                isinstance(t, ast.Name) and t.id == variable_name
                for t in node.targets
            ):
                value = node.value

        elif isinstance(node, ast.AnnAssign):
            if (
                isinstance(node.target, ast.Name)
                and node.target.id == variable_name
            ):
                value = node.value

        if value is not None:
            matches.append(value)

    if not matches:
        raise RuntimeError(f"Atribuição {variable_name} não encontrada.")

    node = matches[-1]

    if (
        isinstance(node, ast.Call)
        and isinstance(node.func, ast.Attribute)
        and node.func.attr == "dedent"
        and node.args
    ):
        result = textwrap.dedent(ast.literal_eval(node.args[0]))
    else:
        result = ast.literal_eval(node)

    if not isinstance(result, str) or not result.strip():
        raise RuntimeError(f"Conteúdo inválido: {variable_name}")

    return result

gt_conversion = extract_assignment(CELL18, "GT_CONVERSION")
test_gt = extract_assignment(CELL18, "TEST_GT")
test_integration = extract_assignment(CELL18, "TEST_INTEGRATION")
protocol = extract_assignment(CELL18, "protocol")

# 2. Backup preventivo dos destinos já existentes
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
backup = DRIVE / "artifacts/backups" / f"l2_restore_{stamp}"
backup.mkdir(parents=True, exist_ok=True)

files = {
    "src/agri_intelligence/data/gt_conversion.py": gt_conversion,
    "tests/unit/test_gt_conversion.py": test_gt,
    "tests/unit/test_l2_integration.py": test_integration,
    "docs/PHASE_2_5_PROTOCOL.md": protocol,
}

for relative, content in files.items():
    destination = REPO / relative

    if destination.exists():
        saved = backup / relative
        saved.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(destination, saved)

    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_text(content.rstrip() + "\n", encoding="utf-8")
    ast.parse(content) if relative.endswith(".py") else None
    print("RESTORED:", relative)

# 3. Confirmar integridade dos arquivos L2
required = [
    REPO / "src/agri_intelligence/data/spatial_split.py",
    REPO / "src/agri_intelligence/data/gt_conversion.py",
    REPO / "tests/unit/test_spatial_split.py",
    REPO / "tests/unit/test_gt_conversion.py",
    REPO / "tests/unit/test_l2_integration.py",
    REPO / "docs/PHASE_2_5_PROTOCOL.md",
]

missing = [str(p) for p in required if not p.is_file()]
if missing:
    raise RuntimeError("Arquivos ausentes após restauração:\n" + "\n".join(missing))

# 4. Compilação
compile_result = subprocess.run(
    [
        sys.executable, "-m", "compileall", "-q",
        "src/agri_intelligence/data",
        "tests/unit"
    ],
    cwd=REPO,
    capture_output=True,
    text=True
)

print("\nCOMPILEALL:", "PASS" if compile_result.returncode == 0 else "FAIL")
if compile_result.stderr:
    print(compile_result.stderr)

if compile_result.returncode != 0:
    raise RuntimeError("Falha na compilação.")

# 5. Suíte L2
env = os.environ.copy()
env["PYTHONPATH"] = (
    str(REPO / "src") + os.pathsep + env.get("PYTHONPATH", "")
)

result = subprocess.run(
    [
        sys.executable, "-m", "pytest", "-q",
        "tests/unit/test_spatial_split.py",
        "tests/unit/test_gt_conversion.py",
        "tests/unit/test_l2_integration.py"
    ],
    cwd=REPO,
    env=env,
    capture_output=True,
    text=True
)

print("\n" + "=" * 65)
print("PHASE 2.5 — L2 TEST SUITE")
print("=" * 65)
print(result.stdout)
if result.stderr:
    print(result.stderr)
print("PYTEST RETURN CODE:", result.returncode)

# 6. Salvar evidência no Drive
manifest = {
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "branch": subprocess.run(
        ["git", "branch", "--show-current"],
        cwd=REPO, capture_output=True, text=True, check=True
    ).stdout.strip(),
    "head": subprocess.run(
        ["git", "rev-parse", "HEAD"],
        cwd=REPO, capture_output=True, text=True, check=True
    ).stdout.strip(),
    "restored_files": [str(p.relative_to(REPO)) for p in required],
    "compileall_returncode": compile_result.returncode,
    "pytest_returncode": result.returncode,
    "pytest_output": result.stdout,
    "backup": str(backup),
    "training_executed": False,
    "push_executed": False
}

manifest_path = (
    DRIVE / "artifacts/manifests" / f"l2_suite_{stamp}.json"
)
manifest_path.parent.mkdir(parents=True, exist_ok=True)
manifest_path.write_text(
    json.dumps(manifest, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("\nEVIDENCE:", manifest_path)
print("\nGIT STATUS:")
print(subprocess.run(
    ["git", "status", "--short"],
    cwd=REPO, capture_output=True, text=True, check=True
).stdout)

if result.returncode != 0:
    raise RuntimeError(
        "A suíte L2 falhou. Não criar commit ainda; "
        "a saída do pytest identifica o próximo ajuste."
    )

print("\nPHASE 2.5 L2 RESTORATION AND TESTS: PASS")

RESTORED: src/agri_intelligence/data/gt_conversion.py
RESTORED: tests/unit/test_gt_conversion.py
RESTORED: tests/unit/test_l2_integration.py
RESTORED: docs/PHASE_2_5_PROTOCOL.md

COMPILEALL: PASS

PHASE 2.5 — L2 TEST SUITE
.................                                                        [100%]
17 passed in 0.85s

PYTEST RETURN CODE: 0

EVIDENCE: /content/drive/MyDrive/ArqTechLabs/Agri-Intelligence/artifacts/manifests/l2_suite_20261110T065844Z.json

GIT STATUS:
?? docs/PHASE_2_5_PROTOCOL.md
?? src/agri_intelligence/data/gt_conversion.py
?? src/agri_intelligence/data/spatial_split.py
?? tests/unit/


PHASE 2.5 L2 RESTORATION AND TESTS: PASS


In [14]:

# =====================================================================
# AGRI-INTELLIGENCE — PHASE 2.5
# FULL VALIDATION + RECOVERY CHECKPOINT
#
# Objetivo:
#   1. Validar recuperação L2
#   2. Executar testes do projeto
#   3. Consolidar checkpoint Git
#   4. Registrar evidências no Drive
#
# Não executa:
#   - treinamento
#   - download de datasets
#   - instalação automática
#   - push
#   - alterações em main
# =====================================================================

from pathlib import Path
from datetime import datetime, timezone
import subprocess
import sys
import os
import json
import hashlib
import ast

# ---------------------------------------------------------------------
# 1. CONFIGURAÇÃO
# ---------------------------------------------------------------------

REPO = Path("/content/agri-intelligence-platform-v1")
DRIVE = Path("/content/drive/MyDrive/ArqTechLabs/Agri-Intelligence")

EXPECTED_BRANCH = "feat/platform-notebook-v1"

L2_FILES = [
    "src/agri_intelligence/data/spatial_split.py",
    "src/agri_intelligence/data/gt_conversion.py",
    "tests/unit/test_spatial_split.py",
    "tests/unit/test_gt_conversion.py",
    "tests/unit/test_l2_integration.py",
    "docs/PHASE_2_5_PROTOCOL.md",
]

assert (REPO / ".git").exists(), "Repositório não encontrado."
assert DRIVE.exists(), "Drive não montado."

os.chdir(REPO)

def run(command, env=None, check=False):
    result = subprocess.run(
        command,
        cwd=REPO,
        env=env,
        capture_output=True,
        text=True
    )

    print("\n$", " ".join(map(str, command)))
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)

    if check and result.returncode != 0:
        raise RuntimeError(
            f"Comando falhou com código {result.returncode}: {command}"
        )

    return result

def git(*args):
    result = subprocess.run(
        ["git", *args],
        cwd=REPO,
        capture_output=True,
        text=True,
        check=True
    )
    return result.stdout.strip()

# ---------------------------------------------------------------------
# 2. IDENTIDADE DO CHECKPOINT
# ---------------------------------------------------------------------

branch = git("branch", "--show-current")
base_head = git("rev-parse", "HEAD")

print("=" * 72)
print("AGRI-INTELLIGENCE — FULL VALIDATION")
print("=" * 72)
print("Branch:", branch)
print("HEAD:", base_head)

if branch != EXPECTED_BRANCH:
    raise RuntimeError(
        f"Branch incorreta: {branch}. Esperada: {EXPECTED_BRANCH}"
    )

# Evita incorporar alterações previamente staged que não pertencem
# a esta recuperação.
staged = run(["git", "diff", "--cached", "--name-only"])

if staged.stdout.strip():
    raise RuntimeError(
        "Há alterações previamente staged. "
        "Não vou incorporá-las ao checkpoint automaticamente:\n"
        + staged.stdout
    )

# ---------------------------------------------------------------------
# 3. EXISTÊNCIA E INTEGRIDADE SINTÁTICA DOS ARQUIVOS L2
# ---------------------------------------------------------------------

print("\n[1/7] FILE RECOVERY")

missing = []
file_evidence = {}

for relative in L2_FILES:
    path = REPO / relative

    if not path.is_file():
        missing.append(relative)
        print("MISSING:", relative)
        continue

    digest = hashlib.sha256(path.read_bytes()).hexdigest()

    file_evidence[relative] = {
        "size_bytes": path.stat().st_size,
        "sha256": digest
    }

    print("PASS:", relative)

if missing:
    raise RuntimeError(
        "Arquivos L2 ausentes:\n" + "\n".join(missing)
    )

# ---------------------------------------------------------------------
# 4. AST + COMPILEALL
# ---------------------------------------------------------------------

print("\n[2/7] STATIC VALIDATION")

python_files = [
    REPO / relative
    for relative in L2_FILES
    if relative.endswith(".py")
]

for path in python_files:
    ast.parse(path.read_text(encoding="utf-8"), filename=str(path))

print("AST VALIDATION: PASS")

compile_result = run([
    sys.executable, "-m", "compileall", "-q",
    "src/agri_intelligence",
    "tests"
])

compile_pass = compile_result.returncode == 0

print(
    "COMPILEALL:",
    "PASS" if compile_pass else "FAIL"
)

if not compile_pass:
    raise RuntimeError("Compilação falhou. Checkpoint não será criado.")

# ---------------------------------------------------------------------
# 5. TESTES L2
# ---------------------------------------------------------------------

print("\n[3/7] PHASE 2.5 — L2 TESTS")

env = os.environ.copy()
env["PYTHONPATH"] = (
    str(REPO / "src")
    + os.pathsep
    + env.get("PYTHONPATH", "")
)

l2_result = run([
    sys.executable, "-m", "pytest", "-q",
    "tests/unit/test_spatial_split.py",
    "tests/unit/test_gt_conversion.py",
    "tests/unit/test_l2_integration.py"
], env=env)

l2_pass = l2_result.returncode == 0

print("L2 RESULT:", "PASS" if l2_pass else "FAIL")

if not l2_pass:
    raise RuntimeError(
        "A suíte L2 falhou. Não será criado checkpoint."
    )

# ---------------------------------------------------------------------
# 6. CHECKPOINT GIT DA RECUPERAÇÃO
# ---------------------------------------------------------------------

print("\n[4/7] GIT CHECKPOINT")

diff_check = run(["git", "diff", "--check"])

if diff_check.returncode != 0:
    raise RuntimeError("git diff --check falhou.")

# Adiciona somente os arquivos recuperados.
run([
    "git", "add", "--",
    *L2_FILES
], check=True)

staged_check = run(["git", "diff", "--cached", "--check"])

if staged_check.returncode != 0:
    raise RuntimeError("Diff staged contém problemas.")

staged_files = git("diff", "--cached", "--name-only")
print("FILES TO COMMIT:")
print(staged_files)

# Usa a identidade Git já configurada no ambiente.
git_name = git("config", "user.name") if run(
    ["git", "config", "user.name"]
).returncode == 0 else ""

git_email = git("config", "user.email") if run(
    ["git", "config", "user.email"]
).returncode == 0 else ""

commit_result = None
commit_hash = None

if not git_name or not git_email:
    print(
        "\nCOMMIT PENDENTE: identidade Git não configurada. "
        "Os arquivos continuam staged e preservados no Colab."
    )
else:
    commit_result = run([
        "git", "commit",
        "-m",
        "recovery(phase-2.5): restore and validate L2 artifacts"
    ])

    if commit_result.returncode == 0:
        commit_hash = git("rev-parse", "HEAD")
        print("RECOVERY COMMIT:", commit_hash)
    else:
        raise RuntimeError("Falha ao criar commit.")

# ---------------------------------------------------------------------
# 7. SUÍTE COMPLETA DO REPOSITÓRIO
# ---------------------------------------------------------------------

print("\n[5/7] FULL REPOSITORY TEST SUITE")

full_result = run([
    sys.executable, "-m", "pytest", "-q"
], env=env)

full_pass = full_result.returncode == 0

print(
    "FULL SUITE:",
    "PASS" if full_pass else "FAIL"
)

# O checkpoint L2 já está preservado.
# Falhas fora da suíte L2 serão registradas separadamente.

# ---------------------------------------------------------------------
# 8. SITUAÇÃO DO REPOSITÓRIO
# ---------------------------------------------------------------------

print("\n[6/7] REPOSITORY STATE")

final_head = git("rev-parse", "HEAD")
final_status = git("status", "--short")
final_branch = git("branch", "--show-current")

print("BRANCH:", final_branch)
print("HEAD:", final_head)
print("STATUS:")
print(final_status if final_status else "CLEAN")

# ---------------------------------------------------------------------
# 9. REGISTRO PERSISTENTE DE EVIDÊNCIAS
# ---------------------------------------------------------------------

print("\n[7/7] PERSISTENT EVIDENCE")

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")

manifest = {
    "project": "Agri-Intelligence",
    "phase": "2.5",
    "stage": "L2 recovery validation",
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "branch": final_branch,
    "base_head": base_head,
    "final_head": final_head,
    "commit_created": commit_hash is not None,
    "commit_hash": commit_hash,
    "compileall": compile_pass,
    "l2_tests": {
        "passed": l2_pass,
        "returncode": l2_result.returncode,
        "output": l2_result.stdout
    },
    "full_suite": {
        "passed": full_pass,
        "returncode": full_result.returncode,
        "output": full_result.stdout
    },
    "files": file_evidence,
    "git_status": final_status,
    "training_started": False,
    "push_executed": False
}

manifest_dir = DRIVE / "artifacts/manifests"
manifest_dir.mkdir(parents=True, exist_ok=True)

manifest_path = manifest_dir / f"full_validation_{stamp}.json"

manifest_path.write_text(
    json.dumps(manifest, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("MANIFEST:", manifest_path)

# ---------------------------------------------------------------------
# 10. RESULTADO E PRÓXIMO MARCO
# ---------------------------------------------------------------------

print("\n" + "=" * 72)
print("VALIDATION SUMMARY")
print("=" * 72)

print("L2:", "PASS" if l2_pass else "FAIL")
print("COMPILE:", "PASS" if compile_pass else "FAIL")
print("FULL SUITE:", "PASS" if full_pass else "FAIL")
print("COMMIT:", commit_hash or "PENDING")
print("PUSH: NOT EXECUTED")
print("TRAINING: NOT EXECUTED")
print("EVIDENCE:", manifest_path)

if l2_pass and compile_pass and full_pass and commit_hash:
    print("\nCHECKPOINT: PASS")
else:
    print(
        "\nCHECKPOINT: PARTIAL — consulte as evidências acima."
    )

print("\nNEXT STAGE: L3 METHODOLOGY FINALIZATION")

AGRI-INTELLIGENCE — FULL VALIDATION
Branch: feat/platform-notebook-v1
HEAD: d8611bda5765449a9b79fb88d824476172cf89f2

$ git diff --cached --name-only

[1/7] FILE RECOVERY
PASS: src/agri_intelligence/data/spatial_split.py
PASS: src/agri_intelligence/data/gt_conversion.py
PASS: tests/unit/test_spatial_split.py
PASS: tests/unit/test_gt_conversion.py
PASS: tests/unit/test_l2_integration.py
PASS: docs/PHASE_2_5_PROTOCOL.md

[2/7] STATIC VALIDATION
AST VALIDATION: PASS

$ /usr/bin/python3 -m compileall -q src/agri_intelligence tests
COMPILEALL: PASS

[3/7] PHASE 2.5 — L2 TESTS

$ /usr/bin/python3 -m pytest -q tests/unit/test_spatial_split.py tests/unit/test_gt_conversion.py tests/unit/test_l2_integration.py
.................                                                        [100%]
17 passed in 0.35s

L2 RESULT: PASS

[4/7] GIT CHECKPOINT

$ git diff --check

$ git add -- src/agri_intelligence/data/spatial_split.py src/agri_intelligence/data/gt_conversion.py tests/unit/test_spatial_split

In [15]:

import subprocess
from pathlib import Path

REPO = Path("/content/agri-intelligence-platform-v1")

def git(*args):
    return subprocess.run(
        ["git", *args],
        cwd=REPO,
        text=True,
        capture_output=True,
        check=True
    ).stdout.strip()

git("config", "user.name", "edu-moraess")
git("config", "user.email", "199427299+edu-moraess@users.noreply.github.com")

assert git("branch", "--show-current") == "feat/platform-notebook-v1"

staged = git("diff", "--cached", "--name-only")
assert staged.strip(), "Não há arquivos staged para o commit."

subprocess.run(
    [
        "git", "commit", "-m",
        "recovery(phase-2.5): restore and validate L2 artifacts"
    ],
    cwd=REPO,
    check=True
)

print("COMMIT:", git("rev-parse", "HEAD"))
print("BRANCH:", git("branch", "--show-current"))
print("STATUS:", git("status", "--short") or "CLEAN")

COMMIT: 0f346a34a69da2ea441e5028cbe3d9e68d36a4c5
BRANCH: feat/platform-notebook-v1
STATUS: CLEAN


In [16]:

from pathlib import Path
import json

ROOT = Path("/content/drive/MyDrive/ArqTechLabs/Agri-Intelligence")
MANIFESTS = ROOT / "artifacts/manifests"
NOTEBOOK = ROOT / "artifacts/backups/notebook_checkpoints/Agri_Intelligence_Platform_v1_20261110T024531Z.ipynb"

print("=" * 75)
print("AGRI-INTELLIGENCE — L3 CONTRACT EXTRACTION")
print("=" * 75)

names = [
    "l3_spatial_preflight.json",
    "l3_pairing_overlap_audit.json",
    "l3_tile_gt_audit.json",
    "l3_selected_blocks.json",
    "l3_stratified_selection.json",
    "l3_optimized_selection.json",
    "l3_final_candidate_selection.json",
]

for name in names:
    path = MANIFESTS / name
    print(f"\n--- {name} ---")
    if not path.exists():
        print("MISSING")
        continue
    try:
        data = json.loads(path.read_text(encoding="utf-8"))
        print("ROOT TYPE:", type(data).__name__)
        if isinstance(data, dict):
            for key, value in data.items():
                if isinstance(value, list):
                    print(f"{key}: list[{len(value)}]")
                    if value:
                        print("  FIRST:", json.dumps(value[0], ensure_ascii=False, default=str)[:1800])
                elif isinstance(value, dict):
                    print(f"{key}: dict keys={list(value.keys())[:40]}")
                else:
                    print(f"{key}: {str(value)[:500]}")
        elif isinstance(data, list):
            print("LENGTH:", len(data))
            if data:
                print("FIRST:", json.dumps(data[0], ensure_ascii=False, default=str)[:1800])
    except Exception as exc:
        print("READ ERROR:", repr(exc))

print("\n" + "=" * 75)
print("NOTEBOOK — CÉLULAS RELACIONADAS À L3")
print("=" * 75)

if NOTEBOOK.exists():
    nb = json.loads(NOTEBOOK.read_text(encoding="utf-8"))
    for i, cell in enumerate(nb.get("cells", [])):
        if cell.get("cell_type") != "code":
            continue
        source = "".join(cell.get("source", []))
        terms = (
            "l3_", "candidate", "selected_blocks", "stratified",
            "tile_gt", "optimized_selection", "final_candidate",
            "alpha", "spatial_split"
        )
        if any(term in source.lower() for term in terms):
            print(f"\n--- CELL {i} ({len(source)} chars) ---")
            print(source[:7000])
else:
    print("NOTEBOOK BACKUP NOT FOUND")

print("\nEXTRACTION COMPLETE — READ ONLY")

AGRI-INTELLIGENCE — L3 CONTRACT EXTRACTION

--- l3_spatial_preflight.json ---
ROOT TYPE: dict
phase: 2.5
stage: L3_PREFLIGHT
mode: READ_ONLY
checks: dict keys=['three_orthomosaics', 'three_annotation_files', 'all_rasters_4_bands', 'all_rasters_epsg32632', 'all_rasters_resolution_5mm', 'all_vectors_have_crs', 'all_vectors_epsg32632', 'no_empty_or_null_geometries', 'polygon_geometry', 'unique_raster_vector_pairs', 'l2_imports']
overall: False
rasters: list[3]
  FIRST: {"path": "/content/drive/MyDrive/ArqTechLabs/Agri-Intelligence/data/raw/zenodo_15235602/orthomosaic_1.tif", "name": "orthomosaic_1.tif", "width": 24964, "height": 41428, "count": 4, "crs": "EPSG:32632", "bounds": [371817.69945915055, 4954236.188792026, 371942.51945915056, 4954443.328792025], "transform": [0.005000000000000279, 0.0, 371817.69945915055, 0.0, -0.004999999999991907, 4954443.328792025], "resolution": [0.005000000000000279, 0.004999999999991907], "dtypes": ["uint8", "uint8", "uint8", "uint8"], "block_shapes": [[2

In [17]:

from pathlib import Path
import json

ROOT = Path("/content/drive/MyDrive/ArqTechLabs/Agri-Intelligence")
M = ROOT / "artifacts/manifests"

FILES = [
    "l3_spatial_preflight.json",
    "l3_pairing_overlap_audit.json",
    "l3_tile_gt_audit.json",
    "l3_selected_blocks.json",
    "l3_stratified_selection.json",
    "l3_optimized_selection.json",
    "l3_final_candidate_selection.json",
]

print("AGRI-INTELLIGENCE | L3 SCHEMA SUMMARY")
print("=" * 65)

for name in FILES:
    p = M / name
    print(f"\n{name}: ", end="")
    if not p.exists():
        print("MISSING")
        continue

    try:
        d = json.loads(p.read_text(encoding="utf-8"))
        if isinstance(d, dict):
            print("OK | keys:", list(d.keys())[:25])
            for k, v in d.items():
                if isinstance(v, list):
                    print(f"  {k}: {len(v)} registros")
                    if v and isinstance(v[0], dict):
                        print("    campos:", list(v[0].keys())[:20])
                elif isinstance(v, dict):
                    print(f"  {k}: dict ({len(v)} campos)")
        elif isinstance(d, list):
            print(f"OK | {len(d)} registros")
            if d and isinstance(d[0], dict):
                print("  campos:", list(d[0].keys())[:20])
    except Exception as e:
        print("ERRO:", type(e).__name__)

print("\n" + "=" * 65)
print("L3 NOTEBOOK CELLS")
nbp = ROOT / "artifacts/backups/notebook_checkpoints/Agri_Intelligence_Platform_v1_20261110T024531Z.ipynb"

if nbp.exists():
    nb = json.loads(nbp.read_text(encoding="utf-8"))
    for i, c in enumerate(nb.get("cells", [])):
        s = "".join(c.get("source", []))
        if c.get("cell_type") == "code" and any(
            x in s.lower() for x in ["l3_", "selected_blocks", "optimized_selection"]
        ):
            print(f"Cell {i}: {len(s)} caracteres")
else:
    print("Notebook não encontrado")

print("\nREAD ONLY — nenhuma alteração realizada.")

AGRI-INTELLIGENCE | L3 SCHEMA SUMMARY

l3_spatial_preflight.json: OK | keys: ['phase', 'stage', 'mode', 'checks', 'overall', 'rasters', 'vectors', 'pairs', 'next_action']
  checks: dict (11 campos)
  rasters: 3 registros
    campos: ['path', 'name', 'width', 'height', 'count', 'crs', 'bounds', 'transform', 'resolution', 'dtypes', 'block_shapes', 'alpha_band', 'valid_alpha_fraction']
  vectors: 3 registros
    campos: ['path', 'name', 'rows', 'crs', 'columns', 'geometry_types', 'empty_geometries', 'null_geometries', 'invalid_geometries', 'bounds']
  pairs: 3 registros
    campos: ['raster', 'matches', 'unique_match']

l3_pairing_overlap_audit.json: OK | keys: ['stage', 'timestamp_utc', 'pairs', 'overlap_2_3', 'errors', 'status']
  pairs: 3 registros
    campos: ['source_id', 'raster', 'vector', 'annotation_count', 'vector_crs', 'geometry_types', 'geometry_coverage']
  overlap_2_3: dict (11 campos)
  errors: 0 registros

l3_tile_gt_audit.json: OK | keys: ['stage', 'status', 'summary', 't

In [18]:

from pathlib import Path
import json, re

ROOT = Path("/content/drive/MyDrive/ArqTechLabs/Agri-Intelligence")
M = ROOT / "artifacts/manifests"
NB = ROOT / "artifacts/backups/notebook_checkpoints/Agri_Intelligence_Platform_v1_20261110T024531Z.ipynb"

print("L3 CONTRACT — COMPACT EXTRACTION")
print("=" * 60)

# 1. Parâmetros e gates dos manifests
for name in [
    "l3_spatial_preflight.json",
    "l3_pairing_overlap_audit.json",
    "l3_selected_blocks.json",
    "l3_stratified_selection.json",
    "l3_optimized_selection.json",
    "l3_final_candidate_selection.json",
]:
    p = M / name
    if not p.exists():
        continue
    d = json.loads(p.read_text())
    print(f"\n[{name}]")
    for k in ("overall", "status", "next_action", "configuration",
              "algorithm", "method", "summary", "gates",
              "baseline_comparison", "overlap_2_3"):
        if k in d:
            value = d[k]
            s = json.dumps(value, ensure_ascii=False, default=str)
            print(f"{k}: {s[:1800]}")

# 2. Regras e configurações presentes nas células L3
nb = json.loads(NB.read_text())
terms = re.compile(
    r"^(?:\s*)(?:def |class |[A-Z][A-Z0-9_]+\s*=|"
    r".*(?:tile_size|tile_width|tile_height|block_size|"
    r"overlap|stride|alpha|threshold|quota|density|"
    r"spatial|split|crs|transform|rasterize|"
    r"selected_blocks|candidate_positions|"
    r"visible_ge|empty|negative|source_id|"
    r"train|val|test|manifest|GeoDataFrame|"
    r"window|from_bounds|intersect|intersection).*)$",
    re.I
)

print("\n" + "=" * 60)
print("NOTEBOOK — LINHAS RELEVANTES")
for i in range(26, 35):
    if i >= len(nb["cells"]):
        continue
    c = nb["cells"][i]
    if c.get("cell_type") != "code":
        continue
    lines = "".join(c.get("source", [])).splitlines()
    hits = [(n + 1, line.strip()) for n, line in enumerate(lines)
            if terms.search(line) and line.strip() and not line.strip().startswith("#")]
    print(f"\nCELL {i} | {len(lines)} linhas | {len(hits)} correspondências")
    for n, line in hits[:18]:
        print(f"{n}: {line[:190]}")

print("\nREAD ONLY — sem alterações, geração de tiles ou treinamento.")

L3 CONTRACT — COMPACT EXTRACTION

[l3_spatial_preflight.json]
overall: false
next_action: "BLOCKED_PENDING_SOURCE_RESOLUTION"

[l3_pairing_overlap_audit.json]
status: "AUDIT_COMPLETED"
overlap_2_3: {"bounds": [487488.3816192643, 5001159.591911526, 487540.1468950519, 5001237.104014865], "area_m2": 4012.43540620316, "fraction_of_ortho2_bounds": 0.3330532623258839, "fraction_of_ortho3_bounds": 0.20967588974288687, "gt2_annotations_intersecting_overlap": 9701, "gt3_annotations_intersecting_overlap": 6299, "positive_area_intersection_pairs": 0, "iou_statistics": {"minimum": null, "median": null, "p95": null, "maximum": null}, "high_iou_pairs_ge_0_90": 0, "high_iou_pairs_ge_0_50": 0, "intersection_pair_sample": []}

[l3_selected_blocks.json]
status: "L3_SELECTION_COMPLETED"
configuration: {"block_size_m": 20.0, "tile_size_px": 1024, "alpha_valid_threshold": 0.995, "alpha_valid_rule": "alpha > 0", "grid_definition": "ix=floor(x/20), iy=floor(y/20)", "tile_origin_rule": "round(pixel_center - 5

In [19]:

from pathlib import Path
import json

M = Path("/content/drive/MyDrive/ArqTechLabs/Agri-Intelligence/artifacts/manifests")

p = json.loads((M / "l3_spatial_preflight.json").read_text())

print("L3 SOURCE RESOLUTION")
print("=" * 55)

for name, check in p["checks"].items():
    if isinstance(check, dict):
        status = check.get("status", check.get("passed", check.get("available")))
        if status is False or status in ("FAIL", "MISSING", "BLOCKED"):
            print(f"\n{name}: {check}")
    elif check is False:
        print(f"\n{name}: FAIL")

print("\nNEXT ACTION:", p.get("next_action"))

print("\nSOURCE PAIRS")
for pair in p.get("pairs", []):
    print(pair)

print("\nRASTER SOURCES")
for raster in p.get("rasters", []):
    print({
        "name": raster.get("name"),
        "path": raster.get("path"),
        "crs": raster.get("crs"),
        "resolution": raster.get("resolution"),
        "alpha_band": raster.get("alpha_band"),
        "valid_alpha_fraction": raster.get("valid_alpha_fraction"),
    })

L3 SOURCE RESOLUTION

unique_raster_vector_pairs: FAIL

NEXT ACTION: BLOCKED_PENDING_SOURCE_RESOLUTION

SOURCE PAIRS
{'raster': 'orthomosaic_1.tif', 'matches': [], 'unique_match': False}
{'raster': 'orthomosaic_2.tif', 'matches': [], 'unique_match': False}
{'raster': 'orthomosaic_3.tif', 'matches': [], 'unique_match': False}

RASTER SOURCES
{'name': 'orthomosaic_1.tif', 'path': '/content/drive/MyDrive/ArqTechLabs/Agri-Intelligence/data/raw/zenodo_15235602/orthomosaic_1.tif', 'crs': 'EPSG:32632', 'resolution': [0.005000000000000279, 0.004999999999991907], 'alpha_band': 4, 'valid_alpha_fraction': 0.71262642923392}
{'name': 'orthomosaic_2.tif', 'path': '/content/drive/MyDrive/ArqTechLabs/Agri-Intelligence/data/raw/zenodo_15235602/orthomosaic_2.tif', 'crs': 'EPSG:32632', 'resolution': [0.005000000000001011, 0.005000000000009796], 'alpha_band': 4, 'valid_alpha_fraction': 0.6625725205323638}
{'name': 'orthomosaic_3.tif', 'path': '/content/drive/MyDrive/ArqTechLabs/Agri-Intelligence/data/raw/